# Tracking y odometría visual

1. En Colab, selecciona GPU si está disponible.
2. Elige una experiencia en el selector y ejecuta las celdas en orden.
3. Abre el **LINK PÚBLICO** para usarla.
4. Descarga los resultados antes de detener la celda de ejecución.

Para cambiar de experiencia, detén la ejecución anterior, cambia el selector y vuelve a ejecutar las celdas. Solo se ejecuta una aplicación a la vez. Los archivos del entorno son temporales; quien tenga el enlace público puede acceder a la aplicación.

## Etapa 01 · Conceptos

Tracking sigue objetos entre imágenes; la odometría visual estima el movimiento de la cámara. Se complementan para distinguir el movimiento de los objetos del movimiento de quien observa.

Preguntas que uno debe poder responder al finalizar la actividad:

- Qué es la odometría visual, sus pasos y tipos.
- Ventajas y desventajas frente a la odometría de ruedas.
- Relación con VIO y SLAM.
- Al menos dos aplicaciones en visión artificial.
- Qué es tracking, sus diferencias y su relación con la odometría.
- Su uso en realidad aumentada, experiencias inmersivas y VFX/cine.

## Etapa 02 · Aplicaciones y conceptos

Selecciona **Etapa 02 - Tracking 2D**.

1. Explora las animaciones que comparan tracking 2D y estimación 3D.
2. Prueba YOLO con ByteTrack o BoT-SORT sobre un video o una webcam fija.
3. Coloca la línea de conteo y observa IDs, cruces y trayectorias.
4. En video, revisa velocidad media en px/s y mapa de calor. Descarga el CSV de trayectorias.

**Alcance:** la velocidad media se obtiene al terminar el video. La webcam ofrece tracking y conteos, sin velocidad instantánea ni mapa de calor. La conversión a unidades físicas se explora en el radar; la cámara móvil, en odometría.

## Etapa 03 · Monitor Inteligente de Tráfico y Navegación

| Nivel | Selección | Qué hacer |
| --- | --- | --- |
| 1 · Radar con cámara fija | Niveles 1 y 2: Radar y analítica | Carga un MP4, marca P1/P2 sobre el suelo e introduce su distancia real. Puedes medirla manualmente en Google Maps. Configura el límite y observa km/h y alertas. |
| 2 · Analítica de tránsito | La misma ejecución del radar | Descarga CSV, mapa de calor y gráfica. Indica la hora inicial de grabación para agrupar automóviles por hora. |
| 3 · Cámara en movimiento | Nivel 3: Odometría visual | Usa una webcam móvil o video caminando. Ajusta altura y campo de visión. Compara velocidades aparentes y compensadas y observa la trayectoria estimada con ORB. |

**Comparación:** prueba primero un video móvil en el radar y después el mismo video en odometría. Observa un auto estacionado y registra si la compensación reduce su velocidad aparente.

**Límites:** el radar admite MP4 y genera reportes al finalizar el análisis. Su CSV resume cada ID y el mapa muestra zonas, sin clasificar carriles. La perspectiva afecta la escala; la odometría depende del suelo y la calibración y puede acumular error. «Velocidad real» en la interfaz significa velocidad compensada estimada.

In [ ]:
# @title 1. Elegir experiencia
from pathlib import Path
import os
import platform
import shutil
import signal
import socket
import subprocess
import sys
import tempfile

experiencia = "Etapa 02 - Tracking 2D" # @param ["Etapa 02 - Tracking 2D", "Etapa 03 - Niveles 1 y 2: Radar y analítica", "Etapa 03 - Nivel 3: Odometría visual"]
APPLICATIONS = {
    "Etapa 02 - Tracking 2D": "tracking_2d",
    "Etapa 03 - Niveles 1 y 2: Radar y analítica": "radar_trafico",
    "Etapa 03 - Nivel 3: Odometría visual": "odometria_visual",
}
if platform.system() != "Linux" or platform.machine() not in ("x86_64", "AMD64"):
    raise RuntimeError("Este notebook prepara un túnel Linux x86_64. Utiliza Google Colab.")
if experiencia not in APPLICATIONS:
    raise ValueError("Selecciona una experiencia de la lista.")
ROOT = Path(tempfile.mkdtemp(prefix="tracking-odometria-"))
APP = ROOT / APPLICATIONS[experiencia]
INSTALLED_APP = None
print(experiencia)
print("Python:", sys.version.split()[0])

In [ ]:
# @title 2. Cargar las aplicaciones incluidas
# Copia de apps/; actualizar esta celda si cambia el código original.
SOURCES = {
    "odometria_visual": "# -*- coding: utf-8 -*-\n\"\"\"\nOdometría visual con cámara en movimiento (Google Colab).\nPipeline por fotograma:\n  YOLO + ByteTrack (vehículos/personas, se enmascaran como objetos dinámicos)\n  -> ORB (FAST orientado + BRIEF rotado) -> emparejamiento Hamming + test de razón\n  -> matriz esencial + RANSAC -> recoverPose (R, dirección de t)\n  -> escala métrica con el plano del suelo (homografía + altura de la cámara)\n  -> pose acumulada (trayectoria) y velocidad de la cámara\n  -> velocidad de cada vehículo: \"aparente\" (cámara supuesta fija) vs \"real\" (compensada con la odometría).\nUso en Colab:  !pip -q install -r apps/odometria_visual/requirements.txt   y luego   !python apps/odometria_visual/app.py\n\"\"\"\nimport asyncio, os, json, math, queue, re, struct, subprocess, threading, time\nfrom collections import deque\nfrom pathlib import Path\nimport cv2, numpy as np, torch\nfrom fastapi import FastAPI, WebSocket\nfrom fastapi.responses import HTMLResponse\n\nNPOOL = int(os.environ.get(\"NPOOL\", 3)); WEIGHTS = os.environ.get(\"WEIGHTS\", \"yolo11m.pt\")\npool = queue.Queue(); app = FastAPI()\n\ndef load_models():\n    from ultralytics import YOLO\n    for _ in range(NPOOL):\n        m = YOLO(WEIGHTS); m.to(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        m.predict(np.zeros((640, 640, 3), np.uint8), verbose=False, half=torch.cuda.is_available()); pool.put(m)\n\nclass Odo:\n    \"\"\"Odometría visual monocular con escala por plano del suelo.\"\"\"\n    def __init__(s, nf=1500, fov=70., hcam=1.4):\n        s.orb = cv2.ORB_create(nfeatures=nf, fastThreshold=12); s.bf = cv2.BFMatcher(cv2.NORM_HAMMING)\n        s.fov, s.h, s.K, s.Ki, s.prev, s.tp = fov, hcam, None, None, None, None\n        s.T = np.eye(4); s.nrm = np.array([0., 1., 0.]); s.mag = 0.; s.dist = 0.; s.sp = 0.; s.tr = {}\n\n    def step(s, gray, mask, t):\n        h, w = gray.shape\n        if s.K is None:\n            f = w / (2 * math.tan(math.radians(s.fov) / 2)); s.K = np.array([[f, 0, w / 2], [0, f, h / 2], [0, 0, 1.]]); s.Ki = np.linalg.inv(s.K)\n        kp, des = s.orb.detectAndCompute(gray, mask)\n        info = dict(nk=len(kp), nm=0, ni=0, st=\"inicializando\", flow=[])\n        dt = (t - s.tp) if s.tp is not None and t - s.tp > 1e-3 else None\n        if s.prev is not None and des is not None and s.prev[1] is not None and len(kp) > 20:\n            pk, pd = s.prev; ok = False\n            pr = [x for x in s.bf.knnMatch(pd, des, k=2) if len(x) == 2]\n            ms = [a for a, b in pr if a.distance < .75 * b.distance]; info[\"nm\"] = len(ms)\n            if len(ms) >= 30:\n                p1 = np.float32([pk[a.queryIdx].pt for a in ms]); p2 = np.float32([kp[a.trainIdx].pt for a in ms])\n                E, mk = cv2.findEssentialMat(p1, p2, s.K, cv2.RANSAC, .999, 1.0)\n                if E is not None and E.shape[0] >= 3 and E.shape[1] == 3:\n                    _, R, tv, mk2 = cv2.recoverPose(E[:3], p1, p2, s.K, mask=mk)\n                    inl = mk2.ravel() > 0; info[\"ni\"] = int(inl.sum())\n                    if info[\"ni\"] >= 25:\n                        ok = True\n                        sel = inl & (p1[:, 1] > h * .45)                      # mitad inferior = suelo\n                        if sel.sum() >= 15:\n                            H, hm = cv2.findHomography(p1[sel], p2[sel], cv2.RANSAC, 2.0)\n                            if H is not None and hm.sum() >= 12:\n                                k, _, ts, ns = cv2.decomposeHomographyMat(H, s.K)\n                                j = max(range(k), key=lambda i: abs(ns[i][1, 0])); nv = ns[j].ravel(); nv = nv if nv[1] > 0 else -nv\n                                s.nrm = .85 * s.nrm + .15 * nv; s.nrm /= np.linalg.norm(s.nrm)   # normal del suelo\n                                mg = float(np.linalg.norm(ts[j])) * s.h                          # |t| en metros\n                                if mg < 3: s.mag = (.6 * s.mag + .4 * mg) if s.mag else mg\n                        tm = tv.ravel() / (np.linalg.norm(tv) + 1e-9) * s.mag\n                        Tr = np.eye(4); Tr[:3, :3] = R.T; Tr[:3, 3] = -R.T @ tm\n                        s.T = s.T @ Tr; d = float(np.linalg.norm(tm)); s.dist += d\n                        if dt: s.sp = .7 * s.sp + .3 * d / dt\n                    o = np.argsort(-inl.astype(int))[:120]\n                    info[\"flow\"] = [[int(p1[i, 0]), int(p1[i, 1]), int(p2[i, 0]), int(p2[i, 1]), int(inl[i])] for i in o]\n            info[\"st\"] = (\"ok\" if info[\"ni\"] >= 40 else \"débil\") if ok else \"perdido\"\n        s.prev, s.tp = (kp, des), t\n        fw, P = s.T[:3, 2], s.T[:3, 3]\n        info[\"cam\"] = dict(p=[round(float(v), 3) for v in P], yaw=round(math.degrees(math.atan2(fw[0], fw[2])), 1), v=round(s.sp * 3.6, 2), d=round(s.dist, 2))\n        return info\n\n    def ground(s, u, v):\n        \"\"\"Punto 3D (marco de la cámara) del píxel (u,v) suponiendo que está sobre el suelo.\"\"\"\n        r = s.Ki @ np.array([u, v, 1.]); den = float(s.nrm @ r)\n        if den < .08: return None\n        X = r * (s.h / den)\n        return X if X[2] < 80 else None\n\n    def track(s, tid, t, u, v):\n        \"\"\"Devuelve (v_aparente, v_real, posición mundo) en km/h y metros.\"\"\"\n        X = s.ground(u, v)\n        if X is None: return None, None, None\n        P = s.T[:3, :3] @ X + s.T[:3, 3]\n        q = s.tr.setdefault(tid, deque(maxlen=40)); q.append((t, P, X))\n        old = next((e for e in q if t - e[0] <= 1.2), None); pw = [round(float(a), 2) for a in P]\n        if old is None or t - old[0] < .4: return None, None, pw\n        dt = t - old[0]\n        return round(float(np.linalg.norm(X - old[2]) / dt * 3.6), 1), round(float(np.linalg.norm(P - old[1]) / dt * 3.6), 1), pw\n\ndef process(m, od, frame, t, cfg, pers):\n    t0 = time.time(); h, w = frame.shape[:2]\n    r = m.track(frame, persist=pers, tracker=\"bytetrack.yaml\", conf=.3, classes=[0, 2, 3, 5, 7], imgsz=640,\n                half=torch.cuda.is_available(), verbose=False)[0]\n    mask = np.full((h, w), 255, np.uint8); dets = []; b = r.boxes\n    if b is not None and b.id is not None:\n        for tid, bx, cl in zip(b.id.int().tolist(), b.xyxy.cpu().numpy(), b.cls.int().tolist()):\n            x1, y1, x2, y2 = [int(v) for v in bx]; cv2.rectangle(mask, (x1 - 8, y1 - 8), (x2 + 8, y2 + 8), 0, -1); dets.append((tid, bx, m.names[cl]))\n    info = od.step(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), mask, t); out = []\n    for tid, bx, name in dets:\n        vn, vc, p = od.track(tid, t, (bx[0] + bx[2]) / 2, bx[3])\n        out.append(dict(bx=[int(v) for v in bx], id=tid, n=name, vn=vn, vc=vc, p=p, mv=bool(vc is not None and vc > 5), al=bool(vc is not None and vc > float(cfg[\"limit\"]))))\n    info.update(b=out, ms=round((time.time() - t0) * 1000), w=w, h=h); return info\n\n@app.websocket(\"/ws/vo\")\nasync def vo(ws: WebSocket):\n    await ws.accept()\n    try: m = pool.get_nowait()\n    except queue.Empty:\n        await ws.send_json({\"error\": \"Todos los espacios de GPU están ocupados. Inténtalo de nuevo en unos segundos.\"}); await ws.close(); return\n    cfg = dict(fov=70, hcam=1.4, nf=1500, limit=30); od = None; first = True\n    try:\n        while True:\n            msg = await ws.receive()\n            if msg[\"type\"] == \"websocket.disconnect\": break\n            if msg.get(\"text\"):\n                cfg.update(json.loads(msg[\"text\"]))\n                if od: od.h = float(cfg[\"hcam\"])\n                continue\n            data = msg.get(\"bytes\") or b\"\"\n            if len(data) < 16: continue\n            t = struct.unpack(\"<d\", data[:8])[0]; frame = cv2.imdecode(np.frombuffer(data[8:], np.uint8), cv2.IMREAD_COLOR)\n            if frame is None: continue\n            if od is None: od = Odo(int(cfg[\"nf\"]), float(cfg[\"fov\"]), float(cfg[\"hcam\"]))\n            res = await asyncio.to_thread(process, m, od, frame, t, cfg, not first); first = False\n            await ws.send_json(res)\n    except Exception:\n        pass\n    finally:\n        pool.put(m)\n\n@app.get(\"/api/info\")\ndef info(): return {\"gpu\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\", \"free\": pool.qsize(), \"slots\": NPOOL}\n\n@app.get(\"/\", response_class=HTMLResponse)\ndef home(): return HTML\n\nHTML = r\"\"\"<!doctype html><html lang=\"es\"><head><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">\n<title>Odometría visual con cámara en movimiento</title>\n<link href=\"https://fonts.googleapis.com/css2?family=Manrope:wght@400;600;800&display=swap\" rel=\"stylesheet\">\n<script src=\"https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js\"></script>\n<script src=\"https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js\"></script>\n<script src=\"https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js\"></script>\n<style>\n:root{--bg:#0f1226;--pn:#171c3a;--ln:#283060;--tx:#e8ebff;--mu:#9aa3d6;--ac:#7cc4ff;--am:#ffb86b;--ok:#7be0a8;--bad:#ff7a8a}\n*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--tx);font:15px/1.55 Manrope,system-ui,sans-serif;padding:28px 32px;max-width:1500px;margin:auto}\nh1{font-size:28px;font-weight:800;margin:0 0 4px}.lead{color:var(--mu);margin:0 0 20px;max-width:80ch}.card{background:var(--pn);border:1px solid var(--ln);border-radius:16px;padding:16px}\n.ctl{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px;align-items:end}label{font-size:12px;color:var(--mu);display:block;margin-bottom:4px}\nselect,input[type=number],input[type=file]{width:100%;background:#0f1330;color:var(--tx);border:1px solid var(--ln);border-radius:10px;padding:9px}input[type=range]{width:100%;accent-color:var(--am)}\n.btn{all:unset;cursor:pointer;background:var(--am);color:#2a1700;font-weight:800;padding:12px 20px;border-radius:12px;text-align:center}.btn:disabled{opacity:.4;cursor:default}.btn.sec{background:#283060;color:var(--tx)}\n.tabs{display:flex;gap:8px;margin:14px 0 10px}.tabs button{all:unset;cursor:pointer;padding:9px 16px;border-radius:10px;border:1px solid var(--ln);font-weight:600;color:var(--mu)}.tabs button.on{background:var(--ac);color:#0c1230;border-color:var(--ac)}\n.g2{display:grid;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));gap:18px;margin:18px 0}canvas.v{width:100%;display:block;border-radius:12px;background:#000}#cv3{width:100%;height:100%;min-height:340px;display:block;border-radius:12px}\n.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(125px,1fr));gap:12px;margin:18px 0}.kpi b{display:block;font-size:26px;font-weight:800}.kpi span{color:var(--mu);font-size:12px}\ntable{width:100%;border-collapse:collapse;font-size:13px}td,th{padding:7px 8px;border-bottom:1px solid var(--ln);text-align:left}th{color:var(--mu);font-weight:600}\na.dl{color:var(--ac);font-weight:600;margin-right:16px}.chk label{display:inline-flex;gap:5px;margin-right:14px;color:var(--tx);font-size:13px}ol{margin:6px 0 0;padding-left:20px;color:#cfd5ff}li{margin-bottom:4px}\n@media(max-width:700px){body{padding:16px}.g2{grid-template-columns:1fr}}\n</style></head><body>\n<h1>Odometría visual con cámara en movimiento</h1>\n<p class=\"lead\">Etapa 03, Nivel 3. Mueve la cámara (o sube un video caminando) y la app estima su trayectoria con ORB y RANSAC. Con esa odometría corrige la velocidad de los vehículos: los autos estacionados dejan de parecer que se mueven.</p>\n<div class=\"card\"><div class=\"tabs\"><button class=\"on\" data-m=\"cam\">Cámara en vivo</button><button data-m=\"file\">Video grabado</button></div>\n<div class=\"ctl\"><div id=\"pcam\"><label>Cámara (en el celular se usa la trasera por defecto)</label><select id=\"cam\"><option value=\"\">Cámara predeterminada</option></select></div>\n<div id=\"pfile\" style=\"display:none\"><label>Video caminando por el pasillo o la calle</label><input type=\"file\" id=\"file\" accept=\"video/*\"><label style=\"margin-top:6px\">Velocidad de reproducción</label><select id=\"rate\"><option value=\"0.5\">0.5x (más preciso)</option><option value=\"1\">1x</option></select></div>\n<div><label>Campo de visión horizontal: <b id=\"fv_\">70</b>°</label><input type=\"range\" id=\"fov\" min=\"40\" max=\"110\" value=\"70\"></div>\n<div><label>Altura de la cámara sobre el suelo (m)</label><input type=\"number\" id=\"hc\" min=\"0.3\" max=\"3\" step=\"0.05\" value=\"1.4\"></div>\n<div><label>Puntos ORB por cuadro: <b id=\"nf_\">1500</b></label><input type=\"range\" id=\"nf\" min=\"500\" max=\"3000\" step=\"100\" value=\"1500\"></div>\n<div><label>Límite de velocidad (km/h)</label><input type=\"number\" id=\"lim\" min=\"5\" max=\"120\" value=\"30\"></div></div>\n<div style=\"display:flex;gap:12px;margin-top:16px;flex-wrap:wrap;align-items:center\"><button class=\"btn\" id=\"go\">Iniciar odometría</button><button class=\"btn sec\" id=\"stp\" disabled>Detener y guardar</button>\n<div class=\"chk\"><label><input type=\"checkbox\" id=\"sf\" checked>Mostrar puntos y flujo</label><label><input type=\"checkbox\" id=\"fl\" checked>Seguir cámara en el mapa</label></div><span id=\"err\" style=\"color:var(--bad)\"></span><span id=\"gpu\" style=\"color:var(--mu);font-size:12px\"></span></div></div>\n<video id=\"src\" playsinline muted style=\"display:none\"></video>\n<div class=\"kpis\" id=\"kp\"></div>\n<div class=\"g2\"><div class=\"card\"><b>Cámara y seguimiento</b><canvas class=\"v\" id=\"lc\" style=\"margin-top:10px;min-height:220px\"></canvas></div>\n<div class=\"card\" style=\"display:flex;flex-direction:column\"><div style=\"display:flex;justify-content:space-between;margin-bottom:10px\"><b>Mapa 3D de la trayectoria</b><button class=\"btn sec\" id=\"vr\" style=\"padding:6px 12px;font-size:12px\">Restablecer vista</button></div><div style=\"flex:1;min-height:340px\"><canvas id=\"cv3\"></canvas></div>\n<div style=\"color:var(--mu);font-size:12px;margin-top:8px\">Naranja: trayectoria de la cámara. Esferas: vehículos (verde = estacionado, naranja = en movimiento, rojo = excede el límite). Arrastra para rotar y usa la rueda para acercar.</div></div></div>\n<div class=\"g2\"><div class=\"card\"><b>Velocidad de la cámara (km/h)</b><canvas id=\"c1\"></canvas></div><div class=\"card\"><b>Inliers de RANSAC por cuadro</b><canvas id=\"c2\"></canvas></div></div>\n<div class=\"g2\"><div class=\"card\"><b>Vehículos: velocidad aparente frente a velocidad real</b><table id=\"tb\" style=\"margin-top:8px\"></table><p id=\"dls\" style=\"margin-bottom:0\"></p></div>\n<div class=\"card\"><b>Cómo funciona</b><ol><li>YOLO detecta vehículos y personas; sus cajas se enmascaran porque son objetos dinámicos.</li><li>ORB (FAST orientado + BRIEF rotado) extrae puntos y se emparejan entre cuadros.</li><li>RANSAC sobre la matriz esencial descarta emparejamientos falsos y entrega rotación y dirección del movimiento.</li><li>La escala sale del plano del suelo (homografía de la mitad inferior y la altura de la cámara), porque una sola cámara no la puede medir.</li><li>La pose se acumula para dibujar la trayectoria.</li><li>Cada vehículo se ubica sobre el suelo en 3D. Velocidad aparente: supone la cámara fija. Velocidad real: usa la pose estimada, así que un auto estacionado queda cerca de 0 km/h.</li></ol></div></div>\n<script>\nconst $=s=>document.querySelector(s),$$=s=>[...document.querySelectorAll(s)];\nlet SRC='cam',WS,STRM,RUN=false,LAST=null,TJ=[],CR=[],LF=0,LP=0,LFPS=0,ch1,ch2,LT0=0;const OFF=document.createElement('canvas');\n$$('.tabs button').forEach(b=>b.onclick=()=>{$$('.tabs button').forEach(x=>x.classList.toggle('on',x==b));SRC=b.dataset.m;$('#pcam').style.display=SRC=='cam'?'block':'none';$('#pfile').style.display=SRC=='file'?'block':'none'});\n$('#fov').oninput=e=>$('#fv_').textContent=e.target.value;$('#nf').oninput=e=>$('#nf_').textContent=e.target.value;\nfetch('/api/info').then(r=>r.json()).then(i=>$('#gpu').textContent=i.gpu+' - '+i.free+'/'+i.slots+' espacios libres').catch(()=>{});\nasync function cams(){try{const d=(await navigator.mediaDevices.enumerateDevices()).filter(x=>x.kind=='videoinput'),s=$('#cam'),v=s.value;s.innerHTML='<option value=\"\">Cámara predeterminada</option>'+d.map((x,i)=>'<option value=\"'+x.deviceId+'\">'+(x.label||'Cámara '+(i+1))+'</option>').join('');s.value=v}catch(e){}}cams();\nconst cfg=()=>({fov:+$('#fov').value,hcam:+$('#hc').value,nf:+$('#nf').value,limit:+$('#lim').value});\n['hc','lim'].forEach(id=>$('#'+id).addEventListener('input',()=>{if(WS&&WS.readyState==1)WS.send(JSON.stringify(cfg()))}));\n/* ---- mapa 3D ---- */\nlet R3,S3,C3,OC,TL,CM,TP=new Float32Array(3*30000),TN=0,CMK={};\nfunction init3(){const el=$('#cv3');R3=new THREE.WebGLRenderer({canvas:el,antialias:true});S3=new THREE.Scene();S3.background=new THREE.Color(0x101426);C3=new THREE.PerspectiveCamera(55,1,.1,600);C3.position.set(0,14,12);\n OC=THREE.OrbitControls?new THREE.OrbitControls(C3,el):{update(){},target:new THREE.Vector3()};OC.enableDamping=true;OC.dampingFactor=.08;\n S3.add(new THREE.AmbientLight(0xffffff,.9));const d=new THREE.DirectionalLight(0xffffff,.6);d.position.set(5,10,5);S3.add(d);S3.add(new THREE.GridHelper(200,100,0x2a3160,0x1d2347));\n const g=new THREE.BufferGeometry();g.setAttribute('position',new THREE.BufferAttribute(TP,3));g.setDrawRange(0,0);TL=new THREE.Line(g,new THREE.LineBasicMaterial({color:0xffb86b}));TL.frustumCulled=false;S3.add(TL);\n CM=new THREE.Mesh(new THREE.ConeGeometry(.5,1.4,4),new THREE.MeshStandardMaterial({color:0x7cc4ff}));CM.geometry.rotateX(-Math.PI/2);S3.add(CM);\n new ResizeObserver(rs3).observe(el.parentElement);rs3();(function l(){requestAnimationFrame(l);OC.update();R3.render(S3,C3)})()}\nfunction rs3(){const p=$('#cv3').parentElement,w=p.clientWidth,h=p.clientHeight;if(w&&h){R3.setSize(w,h,false);C3.aspect=w/h;C3.updateProjectionMatrix()}}\n$('#vr').onclick=()=>{const p=CM.position;C3.position.set(p.x,p.y+14,p.z+12);OC.target.copy(p)};\nfunction map3(r){const[x,y,z]=r.cam.p,nx=x,ny=-y,nz=-z,old=CM.position.clone();CM.position.set(nx,ny,nz);CM.rotation.y=-r.cam.yaw*Math.PI/180;\n if(TN<30000){TP.set([nx,ny,nz],TN*3);TN++;TL.geometry.setDrawRange(0,TN);TL.geometry.attributes.position.needsUpdate=true}\n if($('#fl').checked){const d=CM.position.clone().sub(old);C3.position.add(d);OC.target.add(d)}\n const seen=new Set();r.b.forEach(b=>{if(!b.p)return;seen.add(b.id);let m=CMK[b.id];if(!m){m=CMK[b.id]=new THREE.Mesh(new THREE.SphereGeometry(.4,12,12),new THREE.MeshStandardMaterial({color:0x7be0a8}));S3.add(m)}\n  m.position.set(b.p[0],-b.p[1],-b.p[2]);m.userData.l=LF;m.material.color.set(b.al?0xff7a8a:b.mv?0xffb86b:0x7be0a8)});\n for(const id in CMK)if(LF-CMK[id].userData.l>150){S3.remove(CMK[id]);delete CMK[id]}}\ninit3();\n/* ---- captura y envío ---- */\n$('#go').onclick=async()=>{$('#err').textContent='';const v=$('#src');\n try{if(SRC=='cam'){STRM=await navigator.mediaDevices.getUserMedia({video:{deviceId:$('#cam').value?{exact:$('#cam').value}:undefined,facingMode:$('#cam').value?undefined:{ideal:'environment'},width:{ideal:1280},height:{ideal:720}},audio:false});v.srcObject=STRM}\n  else{const f=$('#file').files[0];if(!f){$('#err').textContent='Elige un video primero.';return}STRM=null;v.srcObject=null;v.src=URL.createObjectURL(f);v.playbackRate=+$('#rate').value;v.onended=()=>stop(true)}\n  await v.play()}catch(e){$('#err').textContent='No se pudo abrir la fuente. Permite el acceso a la cámara o prueba con otra.';return}\n cams();TJ=[];CR=[];LF=0;LAST=null;LP=0;LFPS=0;RUN=true;LT0=performance.now();TN=0;TL.geometry.setDrawRange(0,0);for(const id in CMK){S3.remove(CMK[id]);delete CMK[id]}\n $('#go').disabled=true;$('#stp').disabled=false;$('#dls').innerHTML='';$('#tb').innerHTML='';\n [ch1,ch2].forEach(c=>c&&c.destroy());const mk=(id,col)=>new Chart($(id),{type:'line',data:{labels:[],datasets:[{data:[],borderColor:col,pointRadius:0,tension:.3,fill:true,backgroundColor:col+'22'}]},options:{animation:false,plugins:{legend:{display:false}},scales:{x:{ticks:{color:'#9aa3d6',maxTicksLimit:6},grid:{color:'#283060'}},y:{beginAtZero:true,ticks:{color:'#9aa3d6'},grid:{color:'#283060'}}}}});ch1=mk('#c1','#ffb86b');ch2=mk('#c2','#7cc4ff');\n WS=new WebSocket((location.protocol=='https:'?'wss://':'ws://')+location.host+'/ws/vo');\n WS.onopen=()=>{WS.send(JSON.stringify(cfg()));sendF()};\n WS.onmessage=e=>{const r=JSON.parse(e.data);if(r.error){$('#err').textContent=r.error;stop(false);return}onRes(r);sendF()};WS.onclose=()=>{if(RUN)stop(false)};draw()};\n$('#stp').onclick=()=>stop(true);\nfunction sendF(){if(!RUN)return;const v=$('#src');if(WS.readyState!=1||!v.videoWidth)return setTimeout(sendF,50);\n const w=Math.min(640,v.videoWidth);OFF.width=w;OFF.height=Math.round(w*v.videoHeight/v.videoWidth);OFF.getContext('2d').drawImage(v,0,0,OFF.width,OFF.height);\n const t=SRC=='cam'?performance.now()/1000:v.currentTime;OFF.toBlob(b=>{if(RUN&&WS.readyState==1)WS.send(new Blob([new Float64Array([t]),b]))},'image/jpeg',.75)}\nfunction onRes(r){LAST=r;LF++;const n=performance.now(),t=(n-LT0)/1000,c=r.cam,tv=SRC=='cam'?t:$('#src').currentTime;LFPS=LP?LFPS*.85+.15*(1000/(n-LP)):1;LP=n;\n TJ.push([tv.toFixed(3),c.p[0],c.p[1],c.p[2],c.yaw,c.v,r.ni,r.st]);r.b.forEach(b=>{if(b.p)CR.push([tv.toFixed(3),b.id,b.n,b.p[0],b.p[1],b.p[2],b.vn??'',b.vc??'',b.al?'excede':b.mv?'en movimiento':'estacionado'])});\n const col=r.st=='ok'?'var(--ok)':r.st=='débil'?'var(--am)':'var(--bad)';\n $('#kp').innerHTML=[['Velocidad de la cámara (km/h)',c.v],['Distancia recorrida (m)',c.d],['Rumbo (°)',c.yaw],['Puntos ORB',r.nk],['Coincidencias',r.nm],['Inliers RANSAC',r.ni],['Estado','<span style=\"color:'+col+'\">'+r.st+'</span>'],['FPS',LFPS.toFixed(1)]].map(x=>'<div class=\"card kpi\"><b>'+x[1]+'</b><span>'+x[0]+'</span></div>').join('');\n map3(r);\n if(LF%3==0){[[ch1,c.v],[ch2,r.ni]].forEach(([ch,v])=>{const d=ch.data;d.labels.push(tv.toFixed(0)+' s');d.datasets[0].data.push(v);if(d.labels.length>200){d.labels.shift();d.datasets[0].data.shift()}ch.update('none')})}\n if(LF%4==0)$('#tb').innerHTML='<tr><th>ID</th><th>Clase</th><th>Aparente (km/h)</th><th>Real (km/h)</th><th>Estado</th></tr>'+r.b.filter(b=>b.vc!=null).map(b=>'<tr><td>#'+b.id+'</td><td>'+b.n+'</td><td style=\"color:var(--mu)\">'+b.vn+'</td><td><b>'+b.vc+'</b></td><td>'+(b.al?'excede el límite':b.mv?'en movimiento':'estacionado')+'</td></tr>').join('')}\nfunction draw(){const v=$('#src'),c=$('#lc');if(!v.videoWidth){if(RUN)requestAnimationFrame(draw);return}\n c.width=v.videoWidth;c.height=v.videoHeight;const x=c.getContext('2d'),W=c.width,lw=Math.max(2,W/400);x.drawImage(v,0,0,W,c.height);\n if(LAST){const s=W/LAST.w;if($('#sf').checked)LAST.flow.forEach(f=>{x.strokeStyle=f[4]?'#7be0a8':'#ff7a8a';x.lineWidth=lw*.8;x.beginPath();x.moveTo(f[0]*s,f[1]*s);x.lineTo(f[2]*s,f[3]*s);x.stroke()});\n  x.font='600 '+Math.max(13,W/55)+'px Manrope,sans-serif';\n  LAST.b.forEach(b=>{const k=b.al?'#ff5a6e':b.mv?'#ffb86b':'#7be0a8',[a,e,g,h]=b.bx.map(q=>q*s);x.strokeStyle=k;x.fillStyle=k;x.lineWidth=lw*1.5;x.strokeRect(a,e,g-a,h-e);\n   x.fillText('#'+b.id+' '+b.n,a,Math.max(16,e-6-(b.vc!=null?W/50:0)));if(b.vc!=null){x.fillText('aparente '+b.vn+' | real '+b.vc+' km/h',a,Math.max(16,e-6))}})}\n if(RUN)requestAnimationFrame(draw)}\nfunction stop(save){if(!RUN)return;RUN=false;try{WS.close()}catch(e){}STRM&&STRM.getTracks().forEach(t=>t.stop());$('#src').pause();$('#go').disabled=false;$('#stp').disabled=true;\n if(save){const f=(n,h,rows)=>'<a class=\"dl\" download=\"'+n+'\" href=\"'+URL.createObjectURL(new Blob([h+'\\n'+rows.map(r=>r.join(',')).join('\\n')],{type:'text/csv'}))+'\">Descargar '+n+'</a>';\n  $('#dls').innerHTML=f('trayectoria_camara.csv','tiempo_s,x_m,y_m,z_m,rumbo_deg,vel_kmh,inliers,estado',TJ)+f('vehiculos.csv','tiempo_s,id,clase,x_m,y_m,z_m,v_aparente_kmh,v_real_kmh,estado',CR)}}\n</script></body></html>\"\"\"\n\ndef main():\n    import uvicorn\n    server = uvicorn.Server(uvicorn.Config(app, host=\"127.0.0.1\", port=8000, log_level=\"warning\"))\n    thread = threading.Thread(target=server.run, daemon=True)\n    tunnel = None\n    try:\n        print(\"Cargando\", NPOOL, \"instancias de\", WEIGHTS, \"en\", torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\", flush=True)\n        load_models()\n        thread.start()\n        deadline = time.monotonic() + 30\n        while not server.started:\n            if not thread.is_alive() or time.monotonic() > deadline:\n                raise RuntimeError(\"No se pudo iniciar el servidor en el puerto 8000.\")\n            time.sleep(0.1)\n        binary = Path(\"cloudflared\")\n        if not binary.is_file():\n            temporary = binary.with_suffix(\".download\")\n            try:\n                subprocess.run([\"wget\", \"-q\", \"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\", \"-O\", str(temporary)], check=True)\n                temporary.chmod(0o700)\n                temporary.replace(binary)\n            finally:\n                temporary.unlink(missing_ok=True)\n        tunnel = subprocess.Popen([str(binary.resolve()), \"tunnel\", \"--url\", \"http://127.0.0.1:8000\", \"--no-autoupdate\"], stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)\n        for line in tunnel.stderr:\n            match = re.search(r\"https://[-a-z0-9]+\\.trycloudflare\\.com\", line)\n            if match:\n                print(\"\\nLINK PÚBLICO:\", match.group(0), \"\\nDescarga los resultados antes de interrumpir esta celda.\\n\", flush=True)\n            elif \"error\" in line.lower() or \"failed\" in line.lower():\n                print(line.strip(), flush=True)\n        raise RuntimeError(\"El túnel terminó. Ejecuta de nuevo la celda para obtener otro enlace.\")\n    except KeyboardInterrupt:\n        print(\"\\nCerrando el servidor y el túnel…\", flush=True)\n    finally:\n        if tunnel is not None:\n            tunnel.terminate()\n            try:\n                tunnel.wait(timeout=5)\n            except subprocess.TimeoutExpired:\n                tunnel.kill()\n                tunnel.wait()\n            if tunnel.stderr:\n                tunnel.stderr.close()\n        server.should_exit = True\n        if thread.ident is not None:\n            thread.join(timeout=5)\n\n\nif __name__ == \"__main__\":\n    main()\n",
    "radar_trafico": "# -*- coding: utf-8 -*-\n\"\"\"Radar de tráfico · YOLO11n + ByteTrack · archivo independiente para Google Colab.\n\n1. Selecciona un entorno con GPU (opcional) y carga el repositorio en Colab.\n2. Ejecuta en una celda:\n   !pip -q install -r apps/radar_trafico/requirements.txt\n   !python apps/radar_trafico/app.py\n3. Abre el LINK PÚBLICO impreso, carga un MP4 y calibra P1/P2.\n\nEl enlace da acceso a quien lo conozca. Descarga tus resultados antes de cerrar\nColab: /content es temporal. Interrumpir la celda cierra el servidor y el túnel.\nComprobación pequeña, sin modelos ni videos: !python apps/radar_trafico/app.py --check\n\"\"\"\n\nimport asyncio\nimport csv\nimport math\nimport os\nimport re\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.request\nimport uuid\nfrom collections import Counter, deque\nfrom concurrent.futures import ThreadPoolExecutor\nfrom datetime import datetime, timedelta\nfrom fractions import Fraction\nfrom pathlib import Path\nfrom typing import Literal\n\n\ndef pixel_scale(reference, distance, width, height):\n    if not math.isfinite(distance) or not 0 < distance <= 10000:\n        raise ValueError(\"La distancia debe ser mayor que 0 y como máximo 10000 m.\")\n    if len(reference) != 2 or any(\n        len(p) != 2 or any(not math.isfinite(v) or not 0 <= v <= 1 for v in p)\n        for p in reference\n    ):\n        raise ValueError(\"Marca P1 y P2 dentro de la imagen.\")\n    if math.dist(*reference) < 0.01:\n        raise ValueError(\"Separa los dos puntos de referencia.\")\n    a, b = reference\n    return math.hypot((b[0] - a[0]) * width, (b[1] - a[1]) * height) / distance\n\n\nclass SpeedEstimator:\n    \"\"\"Desplazamiento neto en 0,5 s; misma interpolación que el radar original.\"\"\"\n\n    def __init__(self, scale):\n        self.scale = scale\n        self.tracks = {}\n\n    def update(self, key, point, timestamp):\n        samples = self.tracks.get(key)\n        if samples is None or not 0 < timestamp - samples[-1][0] <= 0.5:\n            self.tracks[key] = deque([(timestamp, point)])\n            return None\n        samples.append((timestamp, point))\n        start = timestamp - 0.5\n        while len(samples) > 2 and samples[1][0] <= start:\n            samples.popleft()\n        first_time, first_point = samples[0]\n        if timestamp - first_time < 0.5 - 1e-9:\n            return None\n        next_time, next_point = samples[1]\n        fraction = max(0, min(1, (start - first_time) / (next_time - first_time)))\n        start_point = tuple(a + (b - a) * fraction for a, b in zip(first_point, next_point))\n        # shortcut: una escala local supone profundidad similar; usar homografía para áreas amplias.\n        return math.dist(start_point, point) / self.scale / 0.5 * 3.6\n\n    def expire(self, timestamp):\n        for key in [k for k, samples in self.tracks.items() if timestamp - samples[-1][0] > 0.5]:\n            del self.tracks[key]\n\n\ndef self_check():\n    assert pixel_scale(((0, 0), (1, 1)), 5, 300, 400) == 100\n    estimator = SpeedEstimator(10)\n    assert estimator.update(1, (0, 0), 0) is None\n    assert estimator.update(1, (10, 0), 0.25) is None\n    assert math.isclose(estimator.update(1, (30, 0), 0.75), 14.4)\n    assert estimator.update(1, (40, 0), 2) is None\n    assert not (60 > 60) and 60.1 > 60\n    for points, distance in [(((0, 0), (0, 0)), 5), (((-1, 0), (1, 1)), 5), (((0, 0), (1, 1)), 0)]:\n        try:\n            pixel_scale(points, distance, 300, 400)\n        except ValueError:\n            pass\n        else:\n            raise AssertionError(\"Se aceptó una calibración inválida\")\n    print(\"OK: escala, interpolación, pérdida de seguimiento y límite estricto.\")\n\n\nif __name__ == \"__main__\" and \"--check\" in sys.argv:\n    self_check()\n    sys.exit(0)\n\nimport av\nimport cv2\nimport numpy as np\nfrom fastapi import FastAPI, File, HTTPException, Query, UploadFile\nfrom fastapi.responses import FileResponse, HTMLResponse, JSONResponse, Response\nfrom pydantic import BaseModel, Field, model_validator\n\nROOT = Path(os.environ.get(\"JOBS_DIR\", \"/content/traffic-monitor\"))\nMODEL = ROOT / \"yolo11n.pt\"\nMAX_UPLOAD = 1024**3\nCLASSES = {\"car\": 2, \"motorcycle\": 3, \"bus\": 5, \"truck\": 7}\nNAMES = {\"car\": \"Automóvil\", \"motorcycle\": \"Motocicleta\", \"bus\": \"Autobús\", \"truck\": \"Camión\"}\nFILES = {\"output.mp4\", \"vehicles.csv\", \"heatmap.png\", \"traffic-by-time.png\"}\njobs = {}\nbusy = threading.Lock()\nstopping = threading.Event()\nexecutor = ThreadPoolExecutor(max_workers=1)\napp = FastAPI(title=\"Traffic Monitor · Colab\", docs_url=None, redoc_url=None)\n\n\nclass UploadLimit:\n    def __init__(self, app):\n        self.app = app\n\n    async def __call__(self, scope, receive, send):\n        if scope[\"type\"] != \"http\" or scope[\"path\"] != \"/api/videos\" or scope[\"method\"] != \"POST\":\n            return await self.app(scope, receive, send)\n        maximum = MAX_UPLOAD + 65536\n        try:\n            length = int(dict(scope[\"headers\"]).get(b\"content-length\", b\"0\"))\n        except ValueError:\n            return await JSONResponse({\"detail\": \"Content-Length inválido.\"}, 400)(scope, receive, send)\n        if length > maximum:\n            return await JSONResponse({\"detail\": \"El máximo es 1 GiB por video.\"}, 413)(scope, receive, send)\n        received = 0\n\n        async def limited_receive():\n            nonlocal received\n            message = await receive()\n            if message[\"type\"] == \"http.request\":\n                received += len(message.get(\"body\", b\"\"))\n                if received > maximum:\n                    raise HTTPException(413, \"El máximo es 1 GiB por video.\")\n            return message\n\n        await self.app(scope, limited_receive, send)\n\n\napp.add_middleware(UploadLimit)\n\n\nclass Calibration(BaseModel):\n    reference: tuple[tuple[float, float], tuple[float, float]]\n    distance_m: float = Field(gt=0, le=10000, allow_inf_nan=False)\n    reference_time: float = Field(default=0, ge=0, allow_inf_nan=False)\n    speed_limit: float = Field(default=60, gt=0, le=300, allow_inf_nan=False)\n    confidence: float = Field(default=0.25, ge=0.05, le=0.9, allow_inf_nan=False)\n    classes: list[Literal[\"car\", \"motorcycle\", \"bus\", \"truck\"]] = Field(\n        default_factory=lambda: list(CLASSES), min_length=1\n    )\n    recording_started_at: datetime | None = None\n\n    @model_validator(mode=\"after\")\n    def valid_points(self):\n        pixel_scale(self.reference, self.distance_m, 1, 1)\n        return self\n\n\ndef get_job(key):\n    if not re.fullmatch(r\"[0-9a-f]{32}\", key) or key not in jobs:\n        raise HTTPException(404, \"Video no encontrado.\")\n    return jobs[key]\n\n\ndef directory(key):\n    return ROOT / key\n\n\ndef inspect_video(path):\n    with av.open(str(path)) as container:\n        if \"mp4\" not in container.format.name.split(\",\") or not container.streams.video:\n            raise ValueError(\"Selecciona un MP4 con una pista de video.\")\n        stream = container.streams.video[0]\n        if not stream.width or not stream.height or stream.width * stream.height > 3840 * 2160:\n            raise ValueError(\"Resolución inválida; máximo 3840 × 2160 o equivalente vertical.\")\n        duration = float(stream.duration * stream.time_base) if stream.duration else (container.duration or 0) / av.time_base\n        if duration <= 0 or next(container.decode(stream), None) is None:\n            raise ValueError(\"No se pudo leer el video o su duración.\")\n        return dict(width=stream.width, height=stream.height, duration=duration, fps=float(stream.average_rate or 0))\n\n\ndef reference_jpeg(path, timestamp):\n    with av.open(str(path)) as container:\n        stream = container.streams.video[0]\n        origin = float((stream.start_time or 0) * stream.time_base)\n        container.seek(int((timestamp + origin) / stream.time_base), stream=stream, backward=True)\n        for frame in container.decode(stream):\n            if frame.time is not None and frame.time - origin >= timestamp - 1e-6:\n                ok, data = cv2.imencode(\".jpg\", frame.to_ndarray(format=\"bgr24\"))\n                if ok:\n                    return data.tobytes()\n    raise ValueError(\"No hay un fotograma disponible en ese instante.\")\n\n\n@app.post(\"/api/videos\", status_code=201)\nasync def upload(file: UploadFile = File(...)):\n    key = uuid.uuid4().hex\n    target = directory(key)\n    try:\n        if not file.filename or not file.filename.lower().endswith(\".mp4\"):\n            raise HTTPException(415, \"Selecciona un archivo MP4.\")\n        target.mkdir(parents=True)\n        size = 0\n        with (target / \"input.upload\").open(\"wb\") as stream:\n            while chunk := await file.read(1024**2):\n                size += len(chunk)\n                if size > MAX_UPLOAD:\n                    raise HTTPException(413, \"El máximo es 1 GiB por video.\")\n                await asyncio.to_thread(stream.write, chunk)\n        try:\n            info = await asyncio.to_thread(inspect_video, target / \"input.upload\")\n        except Exception as exc:\n            raise HTTPException(422, f\"Video no válido: {exc}\") from exc\n        (target / \"input.upload\").replace(target / \"input.mp4\")\n        job = dict(id=key, name=file.filename.replace(\"\\\\\", \"/\").split(\"/\")[-1][:255], state=\"ready\", progress=0, **info)\n        jobs[key] = job\n        return job\n    except Exception:\n        (target / \"input.upload\").unlink(missing_ok=True)\n        (target / \"input.mp4\").unlink(missing_ok=True)\n        if target.exists():\n            target.rmdir()\n        raise\n    finally:\n        await file.close()\n\n\n@app.get(\"/api/jobs/{key}\")\ndef status(key: str):\n    return get_job(key).copy()\n\n\n@app.get(\"/api/jobs/{key}/frame\")\ndef frame(key: str, time: float = Query(default=0, ge=0, allow_inf_nan=False)):\n    job = get_job(key)\n    if time >= job[\"duration\"]:\n        raise HTTPException(422, \"El instante está fuera del video.\")\n    try:\n        return Response(reference_jpeg(directory(key) / \"input.mp4\", time), media_type=\"image/jpeg\")\n    except ValueError as exc:\n        raise HTTPException(422, str(exc)) from exc\n\n\n@app.get(\"/api/jobs/{key}/preview\")\ndef preview(key: str):\n    get_job(key)\n    try:\n        return Response((directory(key) / \"preview.jpg\").read_bytes(), media_type=\"image/jpeg\", headers={\"Cache-Control\": \"no-store\"})\n    except FileNotFoundError:\n        raise HTTPException(404, \"Esperando el primer fotograma.\") from None\n\n\n@app.post(\"/api/jobs/{key}/start\")\ndef start(key: str, config: Calibration):\n    job = get_job(key)\n    if config.reference_time >= job[\"duration\"]:\n        raise HTTPException(422, \"El instante está fuera del video.\")\n    if not busy.acquire(blocking=False):\n        raise HTTPException(409, \"Ya hay un análisis activo; espera a que termine.\")\n    try:\n        if job[\"state\"] != \"ready\":\n            raise HTTPException(409, \"Carga el video otra vez para una nueva ejecución.\")\n        job.update(state=\"running\", progress=0)\n        executor.submit(analyze, key, config)\n    except Exception:\n        busy.release()\n        if job[\"state\"] == \"running\":\n            job[\"state\"] = \"ready\"\n        raise\n    return job.copy()\n\n\n@app.get(\"/files/{key}/{name}\")\ndef download(key: str, name: str, download: bool = False):\n    job = get_job(key)\n    if name not in FILES or job[\"state\"] != \"done\":\n        raise HTTPException(404, \"El resultado todavía no está disponible.\")\n    path = directory(key) / name\n    if not path.is_file():\n        raise HTTPException(404, \"Archivo no encontrado.\")\n    return FileResponse(path, filename=f\"{key[:8]}-{name}\" if download else None)\n\n\ndef write_reports(target, tracks, grid, background, config, duration, video_name):\n    from matplotlib.backends.backend_agg import FigureCanvasAgg\n    from matplotlib.figure import Figure\n    from matplotlib.ticker import MaxNLocator\n\n    fields = [\"id\", \"clase\", \"video\", \"primera_aparicion_s\", \"ultima_aparicion_s\", \"fecha_primera_aparicion\", \"observaciones\", \"maxima_kmh\", \"limite_kmh\", \"exceso\"]\n    with (target / \"vehicles.csv\").open(\"w\", newline=\"\", encoding=\"utf-8-sig\") as output:\n        writer = csv.writer(output)\n        writer.writerow(fields)\n        safe_name = \"'\" + video_name if video_name.lstrip().startswith((\"=\", \"+\", \"-\", \"@\")) else video_name\n        for key, track in tracks.items():\n            date = (config.recording_started_at + timedelta(seconds=track[\"first\"])).isoformat() if config.recording_started_at else \"\"\n            writer.writerow([key, NAMES[track[\"class\"]], safe_name, track[\"first\"], track[\"last\"], date, track[\"observations\"], track[\"maximum\"], config.speed_limit, track[\"alert\"]])\n\n    height, width = background.shape[:2]\n    figure = Figure(figsize=(max(3.6, min(10, 6 * width / height + 1)), 6), layout=\"constrained\", facecolor=\"#172129\")\n    FigureCanvasAgg(figure)\n    ax = figure.subplots()\n    extent = (0, width, height, 0)\n    ax.imshow(cv2.cvtColor(background, cv2.COLOR_BGR2RGB), extent=extent)\n    density = cv2.GaussianBlur(grid, (0, 0), 2)\n    if density.max() > 0:\n        density /= density.max()\n        overlay = ax.imshow(np.ma.masked_where(density < 0.01, density), extent=extent, cmap=\"inferno\", alpha=0.25 + 0.6 * density, vmin=0, vmax=1)\n        colorbar = figure.colorbar(overlay, ax=ax, fraction=0.045, pad=0.04)\n        colorbar.set_ticks([0, 0.5, 1], labels=[\"Baja\", \"Media\", \"Alta\"])\n        colorbar.ax.tick_params(colors=\"#9aabb8\", labelsize=9)\n        colorbar.outline.set_edgecolor(\"#2a3943\")\n    else:\n        ax.text(0.5, 0.5, \"Sin trayectorias\", transform=ax.transAxes, ha=\"center\", color=\"white\", bbox=dict(facecolor=\"black\"))\n    ax.set_title(\"Intensidad relativa de trayectorias\", color=\"#e5edf3\", fontsize=11, pad=12)\n    ax.set_axis_off()\n    figure.savefig(target / \"heatmap.png\", dpi=150, bbox_inches=\"tight\", pad_inches=0.12)\n    figure.clear()\n\n    cars = [track for track in tracks.values() if track[\"class\"] == \"car\"]\n    if config.recording_started_at:\n        origin = config.recording_started_at.replace(minute=0, second=0, microsecond=0)\n        offset = (config.recording_started_at - origin).total_seconds()\n        interval = 3600\n        labels = [(origin + timedelta(hours=i)).isoformat(timespec=\"minutes\") for i in range(int((offset + duration) // interval) + 1)]\n        title = \"Automóviles por hora de grabación\"\n    else:\n        offset = 0\n        interval = 1 if duration < 60 else 60\n        labels = [str(timedelta(seconds=i * interval)) for i in range(int(duration // interval) + 1)]\n        title = \"Automóviles por \" + (\"segundo\" if interval == 1 else \"minuto\") + \" del video\"\n    counts = Counter(int((offset + track[\"first\"]) // interval) for track in cars)\n    figure = Figure(figsize=(10, 5.5), layout=\"constrained\", facecolor=\"#172129\")\n    FigureCanvasAgg(figure)\n    ax = figure.subplots()\n    ax.set_facecolor(\"#172129\")\n    bars = ax.bar(range(len(labels)), [counts[i] for i in range(len(labels))], color=\"#25c2a5\", width=0.6, zorder=3)\n    if len(labels) <= 30:\n        ax.bar_label(bars, labels=[str(counts[i]) if counts[i] else \"\" for i in range(len(labels))], padding=6, color=\"#e5edf3\", fontsize=11)\n    ticks = list(range(0, len(labels), max(1, math.ceil(len(labels) / 10))))\n    ax.set_xticks(ticks, [labels[i].replace(\"T\", \"\\n\") for i in ticks], rotation=25 if len(ticks) > 4 else 0, ha=\"right\" if len(ticks) > 4 else \"center\")\n    ax.yaxis.set_major_locator(MaxNLocator(integer=True))\n    ax.set_ylim(0, max(1, max(counts.values(), default=0)) * 1.2)\n    ax.set_title(title, color=\"#e5edf3\", fontsize=14, pad=18)\n    ax.set_ylabel(\"Automóviles (IDs nuevos)\", color=\"#9aabb8\", labelpad=12)\n    ax.set_xlabel(\"Inicio del intervalo · \" + (\"hora indicada de la cámara\" if config.recording_started_at else \"tiempo transcurrido\"), color=\"#9aabb8\", labelpad=12)\n    ax.tick_params(colors=\"#9aabb8\")\n    ax.grid(axis=\"y\", color=\"#2a3943\", linewidth=0.8, zorder=0)\n    for spine in ax.spines.values():\n        spine.set_visible(False)\n    figure.suptitle(f\"{len(cars)} IDs de automóviles · {duration:.1f} s de video\", color=\"#9aabb8\", fontsize=11)\n    figure.supxlabel(\"Cada ID se cuenta en su primera aparición; los intervalos extremos pueden ser parciales.\", color=\"#9aabb8\", fontsize=9)\n    figure.savefig(target / \"traffic-by-time.png\", dpi=150, bbox_inches=\"tight\")\n    figure.clear()\n\n\ndef analyze(key, config):\n    job, target = jobs[key], directory(key)\n    output = None\n    try:\n        import torch\n        from ultralytics import YOLO\n\n        device = 0 if torch.cuda.is_available() else \"cpu\"\n        job[\"device\"] = torch.cuda.get_device_name(0) if device == 0 else \"CPU\"\n        model = YOLO(str(MODEL))  # Una instancia nueva evita heredar IDs de otra ejecución.\n        scale = pixel_scale(config.reference, config.distance_m, job[\"width\"], job[\"height\"])\n        estimator = SpeedEstimator(scale)\n        tracks, visited = {}, {}\n        factor = min(1, 320 / max(job[\"width\"], job[\"height\"]))\n        grid = np.zeros((max(1, round(job[\"height\"] * factor)), max(1, round(job[\"width\"] * factor))), dtype=np.float32)\n        background = None\n        last_emit, origin, previous = 0, None, None\n        with av.open(str(target / \"input.mp4\")) as source:\n            stream = source.streams.video[0]\n            output = av.open(str(target / \"output.partial.mp4\"), \"w\", options={\"movflags\": \"+faststart\"})\n            encoded = output.add_stream(\"libx264\", rate=stream.average_rate or Fraction(30))\n            encoded.width, encoded.height = stream.width + stream.width % 2, stream.height + stream.height % 2\n            encoded.pix_fmt = \"yuv420p\"\n            encoded.time_base = encoded.codec_context.time_base = Fraction(1, 90000)\n            encoded.options = {\"preset\": \"ultrafast\", \"crf\": \"23\"}\n            for index, frame in enumerate(source.decode(stream)):\n                if stopping.is_set():\n                    raise InterruptedError(\"Análisis interrumpido al cerrar el servidor.\")\n                if frame.pts is None:\n                    raise ValueError(\"El video contiene un fotograma sin marca temporal.\")\n                absolute = frame.pts * frame.time_base\n                if origin is None:\n                    origin = absolute\n                timestamp = float(absolute - origin)\n                if previous is not None and timestamp <= previous:\n                    raise ValueError(\"Las marcas temporales del video no son crecientes.\")\n                previous = timestamp\n                image = frame.to_ndarray(format=\"bgr24\")\n                if background is None:\n                    background = image.copy()\n                result = model.track(image, persist=True, tracker=\"bytetrack.yaml\", device=device, half=device == 0, imgsz=640, conf=config.confidence, classes=[CLASSES[c] for c in config.classes], verbose=False)[0]\n                estimator.expire(timestamp)\n                boxes = result.boxes\n                detected = len(boxes) if boxes is not None else 0\n                if boxes is not None:\n                    ids = boxes.id.int().tolist() if boxes.id is not None else [None] * len(boxes)\n                    for tid, bbox, cls in zip(ids, boxes.xyxy.tolist(), boxes.cls.int().tolist()):\n                        class_name = result.names[cls]\n                        x1, y1, x2, y2 = bbox\n                        point = ((x1 + x2) / 2, y2)\n                        speed = estimator.update(tid, point, timestamp) if tid is not None else None\n                        if tid is not None:\n                            track = tracks.setdefault(tid, dict(first=timestamp, last=timestamp, observations=0, maximum=None, alert=False, **{\"class\": class_name}))\n                            track[\"last\"] = timestamp\n                            track[\"observations\"] += 1\n                            if speed is not None:\n                                track[\"maximum\"] = max(track[\"maximum\"] or 0, speed)\n                                if speed > config.speed_limit and not track[\"alert\"]:\n                                    track.update(alert=True, alert_time=timestamp)\n                            row = min(grid.shape[0] - 1, max(0, int(point[1] / stream.height * grid.shape[0])))\n                            column = min(grid.shape[1] - 1, max(0, int(point[0] / stream.width * grid.shape[1])))\n                            cell = (row, column)\n                            cells = visited.setdefault(tid, set())\n                            if cell not in cells:\n                                cells.add(cell)\n                                grid[cell] += 1\n                        color = (60, 60, 235) if speed is not None and speed > config.speed_limit else (70, 175, 80) if speed is not None else (230, 185, 65)\n                        label = f\"#{tid if tid is not None else '?'} {class_name} \" + (f\"{speed:.1f} km/h\" if speed is not None else \"Midiendo\")\n                        cv2.rectangle(image, (round(x1), round(y1)), (round(x2), round(y2)), color, 2)\n                        cv2.putText(image, label, (round(x1), max(20, round(y1) - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)\n                p1, p2 = [tuple(round(v * size) for v, size in zip(point, (stream.width, stream.height))) for point in config.reference]\n                cv2.line(image, p1, p2, (240, 180, 40), 2, cv2.LINE_AA)\n                cv2.putText(image, f\"Referencia: {config.distance_m:g} m\", p1, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (240, 180, 40), 2)\n                padded = cv2.copyMakeBorder(image, 0, stream.height % 2, 0, stream.width % 2, cv2.BORDER_CONSTANT)\n                annotated = av.VideoFrame.from_ndarray(padded, format=\"bgr24\")\n                annotated.pts, annotated.time_base = round((absolute - origin) * 90000), Fraction(1, 90000)\n                for packet in encoded.encode(annotated):\n                    output.mux(packet)\n                if time.monotonic() - last_emit >= 0.5:\n                    ok, jpeg = cv2.imencode(\".jpg\", image, [cv2.IMWRITE_JPEG_QUALITY, 80])\n                    if not ok:\n                        raise RuntimeError(\"No se pudo generar la vista previa.\")\n                    (target / \"preview.tmp\").write_bytes(jpeg.tobytes())\n                    (target / \"preview.tmp\").replace(target / \"preview.jpg\")\n                    alerts = [dict(id=tid, speed=t[\"maximum\"], time=t[\"alert_time\"], label=NAMES[t[\"class\"]]) for tid, t in tracks.items() if t[\"alert\"]]\n                    maxima = [t[\"maximum\"] for t in tracks.values() if t[\"maximum\"] is not None]\n                    job.update(progress=min(0.99, timestamp / job[\"duration\"]), video_time=timestamp, frames=index + 1, detected=detected, tracked=len(tracks), measured=len(maxima), average=sum(maxima) / len(maxima) if maxima else None, alerts=sorted(alerts, key=lambda item: item[\"time\"], reverse=True)[:20], alert_count=len(alerts))\n                    last_emit = time.monotonic()\n            if background is None:\n                raise ValueError(\"El video no contiene fotogramas decodificables.\")\n            for packet in encoded.encode():\n                output.mux(packet)\n            output.close()\n            output = None\n        (target / \"output.partial.mp4\").replace(target / \"output.mp4\")\n        job[\"state\"] = \"reports\"\n        write_reports(target, tracks, grid, background, config, timestamp, job[\"name\"])\n        maxima = [t[\"maximum\"] for t in tracks.values() if t[\"maximum\"] is not None]\n        alerts = [dict(id=tid, speed=t[\"maximum\"], time=t[\"alert_time\"], label=NAMES[t[\"class\"]]) for tid, t in tracks.items() if t[\"alert\"]]\n        job.update(state=\"done\", progress=1, video_time=timestamp, frames=index + 1, tracked=len(tracks), measured=len(maxima), average=sum(maxima) / len(maxima) if maxima else None, alert_count=len(alerts), alerts=sorted(alerts, key=lambda item: item[\"time\"], reverse=True)[:20])\n    except Exception as exc:\n        job.update(state=\"error\", error=f\"{type(exc).__name__}: {exc}\")\n    finally:\n        if output is not None:\n            try:\n                output.close()\n            except Exception:\n                pass\n        (target / \"output.partial.mp4\").unlink(missing_ok=True)\n        busy.release()\n\n\n@app.get(\"/\", response_class=HTMLResponse)\ndef home():\n    return HTML\n\n\nHTML = r'''<!doctype html>\n<html lang=\"es\"><head><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">\n<title>Radar de tráfico · Colab</title>\n<style>\n:root{color-scheme:dark;--bg:#10171d;--card:#172129;--line:#2a3943;--text:#e5edf3;--muted:#9aabb8;--accent:#25c2a5}\n*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--text);font:15px/1.6 system-ui,sans-serif}\nheader{border-bottom:1px solid var(--line);padding:18px max(18px,calc((100vw - 1440px)/2));background:var(--card)}header strong{font-size:20px}header span{color:var(--muted);margin-left:16px}\nmain{max-width:1480px;margin:auto;padding:28px 20px}h1{font-size:26px;margin:0}h2{font-size:18px;margin:0 0 14px}p{margin:10px 0 18px}.muted,.hint{color:var(--muted)}.hint{font-size:13px}\n.grid{display:grid;grid-template-columns:minmax(0,1fr) 340px;gap:20px;align-items:start}.card{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:20px;min-width:0}.stack{display:grid;gap:16px}\nlabel,legend{display:block;margin:12px 0 6px;font-weight:600;font-size:14px}input:not([type=checkbox]),select{width:100%;padding:10px;border:1px solid var(--line);border-radius:7px;background:var(--bg);color:var(--text);font:inherit}input[type=file]{max-width:560px}\nbutton,.button{display:inline-block;border:1px solid var(--line);border-radius:7px;padding:10px 14px;background:var(--accent);color:#09261f;font:600 14px system-ui;cursor:pointer;text-decoration:none}button.secondary{background:var(--card);color:var(--text)}button:disabled{opacity:.5;cursor:not-allowed}button[aria-pressed=true]{outline:2px solid var(--accent);outline-offset:2px}\n:focus-visible{outline:2px solid var(--accent);outline-offset:3px}.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}.row input[type=number]{width:130px}.error{color:#ff8a92;overflow-wrap:anywhere}.notice{padding:12px;background:#202e37;border-radius:7px}.stage{position:relative;margin:auto;width:100%;background:#080e12}.stage img,.stage svg{display:block;width:100%;height:100%}.stage svg{position:absolute;inset:0;touch-action:none;cursor:crosshair}.stage circle{cursor:grab;stroke:white;stroke-width:3}.stage circle:focus{stroke-width:8;outline:none}.stage text{fill:white;stroke:#10171d;paint-order:stroke;stroke-width:3}\n.viewer{width:100%;max-height:65vh;object-fit:contain;background:#080e12;display:block;border-radius:7px}.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin:18px 0}.kpis b{font-size:28px;display:block}.kpis span{font-size:13px;color:var(--muted)}progress{width:100%;accent-color:var(--accent);height:12px}.alerts{max-height:280px;overflow:auto;padding-left:20px}.alerts li{padding:8px 0;border-bottom:1px solid var(--line)}fieldset{border:0;padding:0;margin:0}.classes{display:flex;flex-wrap:wrap;gap:4px 14px}.classes label{font-weight:400;display:flex;align-items:center;gap:6px}.classes input{accent-color:var(--accent)}.reports{display:grid;grid-template-columns:minmax(0,.85fr) minmax(0,1.15fr);gap:16px;margin-top:18px;align-items:start}.reports img{width:100%;max-height:560px;object-fit:contain;display:block;border-radius:8px}.reports a{display:block}.reports h2{margin-bottom:6px}a{color:var(--accent)}[hidden]{display:none!important}\n@media(max-width:900px){.grid{grid-template-columns:1fr}.reports{grid-template-columns:1fr}.kpis{grid-template-columns:repeat(2,minmax(0,1fr))}header span{display:block;margin:0}main{padding:20px 14px}.card{padding:16px}}\n</style></head><body>\n<header><strong>Traffic Monitor</strong><span>YOLO11n + ByteTrack · Google Colab</span></header>\n<main><h1>Radar de tráfico</h1><p class=\"muted\">Carga un video de una cámara fija, indica una distancia conocida y analiza el tránsito.</p>\n<p class=\"hint\">Estimaciones académicas: la perspectiva cambia la escala. Calibra cerca de los vehículos; no tiene validez para sanciones.</p>\n<section class=\"card\"><label for=\"file\">1. Selecciona un video MP4 (máximo 1 GiB)</label><div class=\"row\"><input id=\"file\" type=\"file\" accept=\"video/mp4,.mp4\"><button id=\"upload\">Cargar video</button></div><p id=\"uploadStatus\" role=\"status\" class=\"hint\"></p></section>\n<p id=\"error\" class=\"error\" role=\"alert\" hidden></p>\n<section id=\"setup\" hidden style=\"margin-top:20px\"><div class=\"grid\">\n<div class=\"card\"><h2>2. Marca P1 y P2 en el suelo</h2><p id=\"videoName\" class=\"hint\"></p>\n<div class=\"stage\" id=\"stage\"><img id=\"frame\" alt=\"Fotograma de la calle para calibrar la distancia\"><svg id=\"overlay\" aria-label=\"Puntos de referencia sobre el video\"><line id=\"referenceLine\" stroke=\"#25c2a5\" stroke-width=\"3\" vector-effect=\"non-scaling-stroke\"/><g><circle data-point=\"0\" tabindex=\"0\" role=\"button\" aria-label=\"Punto 1. Usa las flechas para moverlo.\"/><text id=\"p1text\">P1</text></g><g><circle data-point=\"1\" tabindex=\"0\" role=\"button\" aria-label=\"Punto 2. Usa las flechas para moverlo.\"/><text id=\"p2text\">P2</text></g></svg></div>\n<div class=\"row\" style=\"margin-top:18px\"><button type=\"button\" id=\"select1\" class=\"secondary\" aria-pressed=\"true\">Marcar P1</button><button type=\"button\" id=\"select2\" class=\"secondary\" aria-pressed=\"false\">Marcar P2</button></div>\n<p class=\"hint\">Haz clic para colocar el punto seleccionado o arrastra un punto. Tab y flechas también permiten ajustarlo; Mayús mueve más rápido.</p>\n<label for=\"frameTime\">Elegir otro momento del video (segundos)</label><div class=\"row\"><input id=\"frameTime\" type=\"number\" min=\"0\" step=\"0.1\" value=\"0\" aria-describedby=\"frameHint\"><button id=\"showFrame\" class=\"secondary\">Mostrar fotograma</button></div>\n<p id=\"frameHint\" class=\"hint\">0 corresponde al inicio. Este campo solo selecciona la imagen para calibrar; no cambia los FPS ni la rapidez del análisis.</p></div>\n<form class=\"card\" id=\"config\"><fieldset id=\"parameters\"><h2>3. Configura el análisis</h2>\n<label for=\"source\">¿Cómo obtendrás la distancia?</label><select id=\"source\"><option value=\"manual\">Manual: conozco o medí la distancia</option><option value=\"maps\">Medir en Google Maps</option></select>\n<p class=\"hint\" id=\"maps\" hidden>Ubica los mismos puntos en la vista satelital. Haz clic derecho en P1, elige «Medir distancia», marca P2 y copia los metros aquí. <a href=\"https://www.google.com/maps\" target=\"_blank\" rel=\"noopener noreferrer\">Abrir Google Maps</a>. No se importa automáticamente.</p>\n<label for=\"distance\">Distancia entre P1 y P2 (metros)</label><input id=\"distance\" type=\"number\" min=\"0.01\" max=\"10000\" step=\"any\" required placeholder=\"Ejemplo: 10\"><p id=\"scale\" class=\"notice\" aria-live=\"polite\">Introduce la distancia real.</p>\n<label for=\"limit\">Límite para alertas (km/h)</label><input id=\"limit\" type=\"number\" min=\"0.01\" max=\"300\" step=\"any\" value=\"60\" required><p class=\"hint\">Solo se alerta al superar este valor. No modifica la velocidad calculada.</p>\n<label for=\"confidence\">Confianza mínima</label><input id=\"confidence\" type=\"number\" min=\"0.05\" max=\"0.9\" step=\"0.01\" value=\"0.25\" required>\n<fieldset><legend>Vehículos a detectar</legend><div class=\"classes\"><label><input type=\"checkbox\" name=\"class\" value=\"car\" checked>Auto</label><label><input type=\"checkbox\" name=\"class\" value=\"motorcycle\" checked>Moto</label><label><input type=\"checkbox\" name=\"class\" value=\"bus\" checked>Bus</label><label><input type=\"checkbox\" name=\"class\" value=\"truck\" checked>Camión</label></div></fieldset>\n<label for=\"recording\">Inicio de grabación (opcional)</label><input id=\"recording\" type=\"datetime-local\"><p class=\"hint\">Fecha y hora local de la cámara para contar autos por hora. Sin este dato, la gráfica usa el tiempo transcurrido del video.</p>\n<button id=\"start\" type=\"submit\" disabled>Iniciar análisis</button><p class=\"hint\">Se analizan todos los fotogramas. Cada ID obtiene velocidad después de 0,5 segundos de seguimiento continuo.</p></fieldset></form>\n</div></section>\n<section id=\"results\" hidden style=\"margin-top:24px\"><h2>Monitor y resultados</h2><p id=\"state\" role=\"status\"></p><progress id=\"progress\" max=\"1\" value=\"0\" aria-label=\"Avance del análisis\"></progress>\n<div class=\"kpis\"><div class=\"card\"><b id=\"tracked\">—</b><span>IDs seguidos</span></div><div class=\"card\"><b id=\"measured\">—</b><span>IDs con velocidad válida</span></div><div class=\"card\"><b id=\"average\">—</b><span>Promedio de máximas (km/h)</span></div><div class=\"card\"><b id=\"alertCount\">0</b><span>IDs con exceso de velocidad</span></div></div>\n<div class=\"grid\"><div class=\"card\"><img id=\"preview\" class=\"viewer\" alt=\"Vista previa con detecciones y velocidades\" hidden><video id=\"output\" class=\"viewer\" controls playsinline hidden aria-label=\"Video analizado\"></video></div><div class=\"card\"><h2>Alertas recientes</h2><p class=\"hint\">Una alerta por ID. Se muestra su máxima velocidad válida.</p><ul id=\"alerts\" class=\"alerts\"></ul></div></div>\n<div id=\"downloads\" class=\"card\" hidden style=\"margin-top:18px\"><h2>Descargar resultados</h2><div id=\"links\" class=\"row\"></div><p class=\"hint\">Descarga los archivos antes de cerrar Colab. Los IDs son trayectorias: un vehículo perdido puede recibir otro ID.</p></div>\n<div id=\"reports\" class=\"reports\" hidden><div class=\"card\"><h2>Mapa de calor</h2><p class=\"hint\">Las zonas más cálidas concentran más trayectorias. Cada ID aporta una visita por celda.</p><a id=\"heatmapFull\" target=\"_blank\" rel=\"noopener noreferrer\" aria-label=\"Abrir mapa de calor a tamaño completo\"><img id=\"heatmap\" alt=\"Mapa de calor de trayectorias\"></a><p class=\"hint\">Pulsa la imagen para verla a tamaño completo.</p></div><div class=\"card\"><h2>Automóviles por intervalo</h2><p class=\"hint\">Conteo por primera aparición de cada ID, no por cantidad de fotogramas. Un vehículo puede recibir otro ID si se pierde su seguimiento.</p><a id=\"trafficFull\" target=\"_blank\" rel=\"noopener noreferrer\" aria-label=\"Abrir gráfica temporal a tamaño completo\"><img id=\"traffic\" alt=\"Gráfica de primeras apariciones de automóviles por intervalo\"></a><p class=\"hint\">Una barra vacía indica que no aparecieron IDs nuevos en ese intervalo.</p></div></div>\n</section></main>\n<script>\nconst $=id=>document.getElementById(id);\nlet job=null,points=[[.5,.35],[.5,.65]],selected=0,drag=null,imageReady=false,locked=false,frameTime=0,pollTimer=null;\nconst number=v=>v==null?'—':Number(v).toLocaleString('es-PE',{maximumFractionDigits:2});\nfunction error(message){$('error').textContent=message||'';$('error').hidden=!message;}\nasync function api(url,options){const response=await fetch(url,options);if(!response.ok){const body=await response.json().catch(()=>null);throw new Error(typeof body?.detail==='string'?body.detail:body?.detail?.[0]?.msg||`Error HTTP ${response.status}`);}return response.json();}\nfunction select(index){selected=index;[0,1].forEach(i=>$('select'+(i+1)).setAttribute('aria-pressed',String(i===index)));}\nfunction draw(){if(!job)return;const w=job.width,h=job.height,coords=points.map(p=>[p[0]*w,p[1]*h]);const line=$('referenceLine');['x1','y1','x2','y2'].forEach((key,i)=>line.setAttribute(key,coords.flat()[i]));document.querySelectorAll('[data-point]').forEach((circle,i)=>{circle.setAttribute('cx',coords[i][0]);circle.setAttribute('cy',coords[i][1]);circle.setAttribute('r',Math.max(w*.012,Math.min(w,h)*.018));const text=$('p'+(i+1)+'text');text.setAttribute('x',coords[i][0]+w*.02);text.setAttribute('y',Math.max(h*.035,coords[i][1]-h*.02));text.style.fontSize=Math.max(w*.025,h*.02)+'px';});const pixels=Math.hypot((points[1][0]-points[0][0])*w,(points[1][1]-points[0][1])*h),distance=Number($('distance').value),valid=distance>0&&distance<=10000&&Math.hypot(points[1][0]-points[0][0],points[1][1]-points[0][1])>=.01;$('scale').textContent=valid?`${number(pixels)} píxeles ÷ ${number(distance)} m = ${number(pixels/distance)} píxeles por metro`:'Introduce una distancia positiva y separa P1 de P2.';$('start').disabled=locked||!imageReady||!valid;}\nfunction position(event){const r=$('overlay').getBoundingClientRect();return [Math.max(0,Math.min(1,(event.clientX-r.left)/r.width)),Math.max(0,Math.min(1,(event.clientY-r.top)/r.height))];}\n$('overlay').onpointerdown=event=>{if(locked||!imageReady)return;const handle=event.target.getAttribute('data-point');drag=handle===null?selected:Number(handle);event.currentTarget.setPointerCapture(event.pointerId);points[drag]=position(event);select(1-drag);draw();};\n$('overlay').onpointermove=event=>{if(drag!==null&&!locked){points[drag]=position(event);draw();}};\n$('overlay').onpointerup=$('overlay').onpointercancel=()=>{drag=null;};\ndocument.querySelectorAll('[data-point]').forEach((circle,i)=>circle.onkeydown=event=>{if(locked||!imageReady||!['ArrowLeft','ArrowRight','ArrowUp','ArrowDown'].includes(event.key))return;event.preventDefault();const step=event.shiftKey?.02:.005;points[i]=[Math.max(0,Math.min(1,points[i][0]+(event.key==='ArrowRight'?step:event.key==='ArrowLeft'?-step:0))),Math.max(0,Math.min(1,points[i][1]+(event.key==='ArrowDown'?step:event.key==='ArrowUp'?-step:0)))];draw();});\n$('select1').onclick=()=>select(0);$('select2').onclick=()=>select(1);$('distance').oninput=draw;\n$('source').onchange=()=>{$('maps').hidden=$('source').value!=='maps';$('distance').value='';draw();};\nfunction loadFrame(timestamp){imageReady=false;frameTime=timestamp;draw();$('frame').src=`/api/jobs/${job.id}/frame?time=${timestamp}`;}\n$('frame').onload=()=>{imageReady=true;draw();};$('frame').onerror=()=>{imageReady=false;draw();error('No se pudo cargar el fotograma. Selecciona otro instante.');};\n$('showFrame').onclick=()=>{const timestamp=Number($('frameTime').value);if(!Number.isFinite(timestamp)||timestamp<0||timestamp>=job.duration){error('El instante está fuera del video.');return;}error('');loadFrame(timestamp);};\n$('upload').onclick=async()=>{const file=$('file').files[0];if(!file){error('Selecciona un MP4.');return;}if(file.size>1024**3){error('El máximo es 1 GiB por video.');return;}error('');$('upload').disabled=true;$('setup').hidden=true;$('results').hidden=true;$('uploadStatus').textContent='Subiendo y validando el video…';try{const body=new FormData();body.append('file',file);job=await api('/api/videos',{method:'POST',body});clearTimeout(pollTimer);locked=false;points=[[.5,.35],[.5,.65]];select(0);$('parameters').disabled=false;$('showFrame').disabled=false;$('frameTime').disabled=false;document.querySelectorAll('[data-point]').forEach(c=>c.setAttribute('tabindex','0'));$('setup').hidden=false;$('videoName').textContent=`${job.name} · ${job.width} × ${job.height} · ${number(job.duration)} s`;$('stage').style.aspectRatio=`${job.width}/${job.height}`;$('stage').style.maxWidth=`${job.width/job.height*65}vh`;$('overlay').setAttribute('viewBox',`0 0 ${job.width} ${job.height}`);$('frameTime').value=0;$('frameTime').max=Math.max(0,job.duration-.001);$('uploadStatus').textContent='Video listo. Marca los puntos y configura el análisis.';loadFrame(0);}catch(e){error(e.message);$('uploadStatus').textContent='No se pudo cargar el video.';}finally{$('upload').disabled=false;}};\n$('config').onsubmit=async event=>{event.preventDefault();if(!job||!imageReady||locked)return;const classes=[...document.querySelectorAll('[name=class]:checked')].map(c=>c.value);if(!classes.length){error('Selecciona al menos una clase de vehículo.');return;}error('');$('start').disabled=true;try{const config={reference:points,distance_m:Number($('distance').value),reference_time:frameTime,speed_limit:Number($('limit').value),confidence:Number($('confidence').value),classes,recording_started_at:$('recording').value||null};await api(`/api/jobs/${job.id}/start`,{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(config)});locked=true;$('parameters').disabled=true;$('showFrame').disabled=true;$('frameTime').disabled=true;document.querySelectorAll('[data-point]').forEach(c=>c.setAttribute('tabindex','-1'));$('upload').disabled=true;$('file').disabled=true;$('results').hidden=false;$('downloads').hidden=true;$('reports').hidden=true;$('output').hidden=true;$('output').removeAttribute('src');$('preview').hidden=true;$('alerts').replaceChildren();['tracked','measured','average'].forEach(id=>$(id).textContent='—');$('alertCount').textContent='0';poll();$('results').scrollIntoView({behavior:'smooth',block:'start'});}catch(e){error(e.message);draw();}};\nasync function poll(){try{const state=await api(`/api/jobs/${job.id}`);$('state').textContent=state.state==='reports'?'Generando informes…':state.state==='done'?'Análisis completado':state.state==='error'?'El análisis terminó con un error':`Procesando en ${state.device||'el dispositivo disponible'} · ${number(state.video_time||0)} / ${number(state.duration)} s`;$('progress').value=state.progress||0;$('tracked').textContent=number(state.tracked);$('measured').textContent=number(state.measured);$('average').textContent=number(state.average);$('alertCount').textContent=number(state.alert_count||0);$('alerts').replaceChildren(...(state.alerts||[]).map(a=>{const li=document.createElement('li');li.textContent=`${a.label} #${a.id}: ${number(a.speed)} km/h · alerta en ${number(a.time)} s`;return li;}));if(state.frames){$('preview').hidden=false;$('preview').src=`/api/jobs/${job.id}/preview?t=${Date.now()}`;}if(state.state==='done'){const base=`/files/${job.id}/`;$('preview').hidden=true;$('output').src=base+'output.mp4';$('output').hidden=false;const files=[['output.mp4','Video anotado'],['vehicles.csv','CSV por vehículo'],['heatmap.png','Mapa de calor'],['traffic-by-time.png','Gráfica temporal']];$('links').replaceChildren(...files.map(([file,label])=>{const a=document.createElement('a');a.className='button';a.textContent=label;a.href=base+file+'?download=true';return a;}));$('heatmap').src=$('heatmapFull').href=base+'heatmap.png';$('traffic').src=$('trafficFull').href=base+'traffic-by-time.png';$('downloads').hidden=false;$('reports').hidden=false;$('upload').disabled=false;$('file').disabled=false;return;}if(state.state==='error'){error(state.error);$('upload').disabled=false;$('file').disabled=false;return;}}catch(e){error('No se pudo consultar el progreso; reintentando. '+e.message);}pollTimer=setTimeout(poll,1000);}\n</script></body></html>'''\n\n\ndef main():\n    import uvicorn\n    import torch\n    ROOT.mkdir(parents=True, exist_ok=True)\n    os.environ.setdefault(\"YOLO_CONFIG_DIR\", str(ROOT / \"ultralytics\"))\n    os.environ.setdefault(\"MPLCONFIGDIR\", str(ROOT / \"matplotlib\"))\n    from ultralytics import YOLO\n\n    print(\"Preparando YOLO11n. Dispositivo:\", torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\", flush=True)\n    YOLO(str(MODEL))\n    server = uvicorn.Server(uvicorn.Config(app, host=\"127.0.0.1\", port=8000, log_level=\"warning\"))\n    thread = threading.Thread(target=server.run, daemon=True)\n    tunnel = None\n    try:\n        thread.start()\n        deadline = time.monotonic() + 30\n        while not server.started:\n            if not thread.is_alive() or time.monotonic() > deadline:\n                raise RuntimeError(\"No se pudo iniciar el servidor en el puerto 8000.\")\n            time.sleep(0.1)\n        binary = ROOT / \"cloudflared\"\n        if not binary.is_file():\n            print(\"Descargando el túnel oficial de Cloudflare…\", flush=True)\n            request = urllib.request.Request(\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\", headers={\"User-Agent\": \"TrafficMonitor\"})\n            temporary = binary.with_suffix(\".download\")\n            try:\n                with urllib.request.urlopen(request, timeout=60) as response, temporary.open(\"wb\") as target:\n                    while chunk := response.read(1024**2):\n                        target.write(chunk)\n                temporary.chmod(0o700)\n                temporary.replace(binary)\n            finally:\n                temporary.unlink(missing_ok=True)\n        tunnel = subprocess.Popen([str(binary.resolve()), \"tunnel\", \"--url\", \"http://127.0.0.1:8000\", \"--no-autoupdate\"], stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)\n        found = False\n        for line in tunnel.stderr:\n            match = re.search(r\"https://[-a-z0-9]+\\.trycloudflare\\.com\", line)\n            if match:\n                found = True\n                print(\"\\nLINK PÚBLICO:\", match.group(0), \"\\nMantén esta celda ejecutándose. Interrúmpela para cerrar.\\n\", flush=True)\n            elif not found and (\"error\" in line.lower() or \"failed\" in line.lower()):\n                print(line.strip(), flush=True)\n        raise RuntimeError(\"El túnel terminó. Ejecuta de nuevo la celda para obtener otro enlace.\")\n    except KeyboardInterrupt:\n        print(\"\\nCerrando el radar y el túnel…\", flush=True)\n    finally:\n        stopping.set()\n        if tunnel is not None:\n            tunnel.terminate()\n            try:\n                tunnel.wait(timeout=5)\n            except subprocess.TimeoutExpired:\n                tunnel.kill()\n                tunnel.wait()\n            if tunnel.stderr:\n                tunnel.stderr.close()\n        server.should_exit = True\n        thread.join(timeout=5)\n        executor.shutdown(wait=True, cancel_futures=True)\n\n\nif __name__ == \"__main__\":\n    main()\n",
    "tracking_2d": "# -*- coding: utf-8 -*-\n\"\"\"\nTracking 2D y comparación con odometría visual (Google Colab)\nVentana 1: por qué la odometría visual / Pose Tracking 3D es más difícil que el tracking 2D (animaciones 3D).\nVentana 2: tracking 2D con YOLO + ByteTrack / BoT-SORT, conteo por línea virtual y análisis de trayectorias.\nUso en Colab:  !pip -q install -r apps/tracking_2d/requirements.txt   y luego   !python apps/tracking_2d/app.py\n\"\"\"\nimport asyncio, os, csv, json, math, time, uuid, queue, shutil, subprocess, threading, re\nfrom collections import deque\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport cv2, numpy as np, torch\nfrom fastapi import FastAPI, UploadFile, File, Form, WebSocket\nfrom fastapi.responses import HTMLResponse, FileResponse, JSONResponse\n\nW = Path(os.environ.get(\"JOBS_DIR\", \"/content/jobs\")); W.mkdir(parents=True, exist_ok=True)\nNPOOL = int(os.environ.get(\"NPOOL\", 6))               # instancias de YOLO en paralelo\nWEIGHTS = os.environ.get(\"WEIGHTS\", \"yolo11l.pt\")\npool, jobs = queue.Queue(), {}\nex = ThreadPoolExecutor(NPOOL)\napp = FastAPI()\nstopping = threading.Event()\n\ndef load_models():\n    from ultralytics import YOLO\n    for _ in range(NPOOL):\n        m = YOLO(WEIGHTS); m.to(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        m.predict(np.zeros((640, 640, 3), np.uint8), verbose=False, half=torch.cuda.is_available())\n        pool.put(m)\n\ndef color(tid):\n    return tuple(int(v) for v in cv2.cvtColor(np.uint8([[[(tid * 47) % 180, 190, 255]]]), cv2.COLOR_HSV2BGR)[0, 0])\n\ndef run(job, tracker, conf, line, classes):\n    d = W / job[\"id\"]; job[\"state\"] = \"queued\"; m = pool.get(); job[\"state\"] = \"running\"\n    vw = None\n    gen = None\n    try:\n        if stopping.is_set(): raise InterruptedError(\"Análisis interrumpido al cerrar el servidor.\")\n        cap = cv2.VideoCapture(str(d / \"in.mp4\")); fps = cap.get(5) or 30; w, h = int(cap.get(3)), int(cap.get(4)); n = max(int(cap.get(7)), 1)\n        ok, f0 = cap.read(); cap.release(); cv2.imwrite(str(d / \"poster.jpg\"), f0)\n        vw = cv2.VideoWriter(str(d / \"raw.mp4\"), cv2.VideoWriter_fourcc(*\"mp4v\"), fps, (w, h))\n        x1, y1, x2, y2 = line[0] * w, line[1] * h, line[2] * w, line[3] * h\n        side = lambda x, y: (x2 - x1) * (y - y1) - (y2 - y1) * (x - x1)\n        last, done, tail, tr, series, pc = {}, set(), {}, [], [], {}\n        cin = cout = 0; names = m.names; t0 = time.time()\n        gen = m.track(str(d / \"in.mp4\"), stream=True, tracker=tracker + \".yaml\", conf=conf, half=torch.cuda.is_available(),\n                      persist=False, verbose=False, classes=classes or None, imgsz=960)\n        for i, r in enumerate(gen):\n            if stopping.is_set(): raise InterruptedError(\"Análisis interrumpido al cerrar el servidor.\")\n            fr = r.orig_img.copy(); b = r.boxes; act = 0\n            if b is not None and b.id is not None:\n                for tid, bx, c in zip(b.id.int().tolist(), b.xyxy.cpu().numpy(), b.cls.int().tolist()):\n                    cx, cy = float((bx[0] + bx[2]) / 2), float((bx[1] + bx[3]) / 2); act += 1\n                    tr.append((tid, c, i, round(cx, 1), round(cy, 1)))\n                    s = side(cx, cy)\n                    if tid in last and last[tid] * s < 0:\n                        dr = \"in\" if s > 0 else \"out\"\n                        if (tid, dr) not in done:\n                            done.add((tid, dr)); pc.setdefault(names[c], [0, 0])[0 if dr == \"in\" else 1] += 1\n                            if dr == \"in\": cin += 1\n                            else: cout += 1\n                    last[tid] = s\n                    col = color(tid); tail.setdefault(tid, deque(maxlen=45)).append((int(cx), int(cy)))\n                    cv2.rectangle(fr, (int(bx[0]), int(bx[1])), (int(bx[2]), int(bx[3])), col, 2)\n                    cv2.putText(fr, f\"#{tid} {names[c]}\", (int(bx[0]), max(18, int(bx[1]) - 6)), 0, .6, col, 2)\n                    cv2.polylines(fr, [np.array(tail[tid], np.int32)], False, col, 2)\n            cv2.line(fr, (int(x1), int(y1)), (int(x2), int(y2)), (80, 180, 255), 3)\n            cv2.rectangle(fr, (0, 0), (260, 70), (30, 20, 16), -1)\n            cv2.putText(fr, f\"Entran: {cin}\", (10, 28), 0, .8, (160, 240, 160), 2); cv2.putText(fr, f\"Salen: {cout}\", (10, 58), 0, .8, (120, 120, 255), 2)\n            vw.write(fr); series.append(act); job[\"p\"] = min((i + 1) / n, .99)\n        vw.release(); dt = time.time() - t0\n        subprocess.run([\"ffmpeg\", \"-y\", \"-i\", str(d / \"raw.mp4\"), \"-vf\", \"scale='min(960,iw)':-2\", \"-vcodec\", \"libx264\", \"-pix_fmt\", \"yuv420p\", \"-preset\", \"veryfast\", \"-crf\", \"28\", \"-movflags\", \"+faststart\", str(d / \"video.mp4\"), \"-loglevel\", \"error\"])\n        per = {}\n        for tid, c, f, x, y in tr: per.setdefault(tid, [c, []])[1].append((f, x, y))\n        rows = []\n        for tid, (c, p) in per.items():\n            dist = sum(math.dist(p[k][1:], p[k + 1][1:]) for k in range(len(p) - 1)); dur = (p[-1][0] - p[0][0] + 1) / fps\n            rows.append(dict(id=tid, cls=names[c], frames=len(p), dur=round(dur, 2), dist=round(dist), vel=round(dist / dur, 1) if dur else 0))\n        rows.sort(key=lambda r: -r[\"dist\"])\n        heat = np.zeros((18, 32), int)\n        for _, _, _, x, y in tr: heat[min(17, int(y / h * 18)), min(31, int(x / w * 32))] += 1\n        with open(d / \"trayectorias.csv\", \"w\", newline=\"\") as f:\n            wr = csv.writer(f); wr.writerow([\"id\", \"clase\", \"frame\", \"tiempo_s\", \"cx\", \"cy\"])\n            for tid, c, fi, x, y in tr: wr.writerow([tid, names[c], fi, round(fi / fps, 3), x, y])\n        st = max(1, len(series) // 300)\n        top = [r[\"id\"] for r in rows[:300]]\n        res = dict(w=w, h=h, fps=round(fps, 1), frames=len(series), ids=len(per), cin=cin, cout=cout, pc=pc, line=line,\n                   inf_fps=round(len(series) / dt, 1), mb=round((d / \"video.mp4\").stat().st_size / 1e6, 1), series=series[::st], st=st, rows=rows[:50], heat=heat.tolist(),\n                   tr=[dict(id=t, p=[[round(a), round(b_)] for _, a, b_ in per[t][1][::max(1, len(per[t][1]) // 150)]]) for t in top])\n        json.dump(res, open(d / \"resultados.json\", \"w\")); job.update(res=res, p=1, state=\"done\")\n    except Exception as e:\n        job.update(state=\"error\", err=str(e))\n    finally:\n        try:\n            if gen is not None: gen.close()\n        finally:\n            if vw is not None: vw.release()\n            pool.put(m)\n\n@app.post(\"/api/track\")\nasync def track(file: UploadFile = File(...), tracker: str = Form(\"bytetrack\"), conf: float = Form(.3),\n                line: str = Form(\"[0,0.5,1,0.5]\"), classes: str = Form(\"\")):\n    for p in W.iterdir():                                     # limpieza de trabajos con más de 2 h\n        if p.is_dir() and time.time() - p.stat().st_mtime > 7200: shutil.rmtree(p, ignore_errors=True)\n    jid = uuid.uuid4().hex[:10]; d = W / jid; d.mkdir()\n    with open(d / \"in.mp4\", \"wb\") as f: shutil.copyfileobj(file.file, f)\n    job = dict(id=jid, state=\"queued\", p=0); jobs[jid] = job\n    ex.submit(run, job, tracker if tracker in (\"bytetrack\", \"botsort\") else \"bytetrack\", conf, json.loads(line),\n              [int(c) for c in classes.split(\",\") if c != \"\"])\n    return {\"id\": jid}\n\n@app.get(\"/api/job/{jid}\")\ndef job_state(jid: str): return jobs.get(jid, {\"state\": \"error\", \"err\": \"Trabajo no encontrado\"})\n\n@app.get(\"/api/info\")\ndef info():\n    return {\"gpu\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\", \"queue\": sum(j[\"state\"] == \"queued\" for j in jobs.values()),\n            \"running\": sum(j[\"state\"] == \"running\" for j in jobs.values()), \"slots\": NPOOL}\n\n@app.get(\"/files/{jid}/{name}\")\ndef files(jid: str, name: str):\n    if name not in (\"video.mp4\", \"poster.jpg\", \"trayectorias.csv\", \"resultados.json\") or not re.fullmatch(r\"[0-9a-f]{10}\", jid): return JSONResponse({}, 404)\n    return FileResponse(W / jid / name)\n\n@app.websocket(\"/ws/live\")\nasync def live(ws: WebSocket):\n    \"\"\"Tracking en tiempo real: el navegador envía fotogramas JPEG y recibe cajas, IDs y conteos.\"\"\"\n    await ws.accept()\n    try: m = pool.get_nowait()\n    except queue.Empty:\n        await ws.send_json({\"error\": \"Todos los espacios de GPU están ocupados. Inténtalo de nuevo en unos segundos.\"}); await ws.close(); return\n    cfg = dict(tracker=\"bytetrack\", conf=.3, line=[0, .5, 1, .5], classes=[]); first = True\n    last, done, pc, cin, cout = {}, set(), {}, 0, 0\n    try:\n        while True:\n            msg = await ws.receive()\n            if msg[\"type\"] == \"websocket.disconnect\": break\n            if msg.get(\"text\"):\n                cm = json.loads(msg[\"text\"])\n                if first and cm.get(\"tracker\") in (\"bytetrack\", \"botsort\"): cfg[\"tracker\"] = cm[\"tracker\"]\n                if \"line\" in cm and cm[\"line\"] != cfg[\"line\"]: last.clear()\n                for k in (\"conf\", \"line\", \"classes\"):\n                    if k in cm: cfg[k] = cm[k]\n                continue\n            frame = cv2.imdecode(np.frombuffer(msg.get(\"bytes\") or b\"\", np.uint8), cv2.IMREAD_COLOR)\n            if frame is None: continue\n            h, w = frame.shape[:2]; t0 = time.time(); pers = not first\n            r = (await asyncio.to_thread(lambda: m.track(frame, persist=pers, tracker=cfg[\"tracker\"] + \".yaml\", conf=float(cfg[\"conf\"]),\n                 half=torch.cuda.is_available(), verbose=False, classes=cfg[\"classes\"] or None, imgsz=640)))[0]\n            first = False\n            x1, y1, x2, y2 = cfg[\"line\"][0] * w, cfg[\"line\"][1] * h, cfg[\"line\"][2] * w, cfg[\"line\"][3] * h\n            out, b = [], r.boxes\n            if b is not None and b.id is not None:\n                for tid, bx, cl in zip(b.id.int().tolist(), b.xyxy.cpu().numpy(), b.cls.int().tolist()):\n                    cx, cy = (bx[0] + bx[2]) / 2, (bx[1] + bx[3]) / 2\n                    sd = (x2 - x1) * (cy - y1) - (y2 - y1) * (cx - x1)\n                    if tid in last and last[tid] * sd < 0:\n                        dr = \"in\" if sd > 0 else \"out\"\n                        if (tid, dr) not in done:\n                            done.add((tid, dr)); pc.setdefault(m.names[cl], [0, 0])[0 if dr == \"in\" else 1] += 1\n                            if dr == \"in\": cin += 1\n                            else: cout += 1\n                    last[tid] = sd\n                    out.append([int(bx[0]), int(bx[1]), int(bx[2]), int(bx[3]), tid, cl, m.names[cl]])\n            await ws.send_json(dict(b=out, cin=cin, cout=cout, pc=pc, ms=round((time.time() - t0) * 1000), w=w, h=h))\n    except Exception:\n        pass\n    finally:\n        pool.put(m)\n\n@app.get(\"/\", response_class=HTMLResponse)\ndef home(): return HTML\n\nHTML = r\"\"\"<!doctype html><html lang=\"es\"><head><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">\n<title>Tracking 2D y trayectorias</title>\n<link href=\"https://fonts.googleapis.com/css2?family=Manrope:wght@400;600;800&display=swap\" rel=\"stylesheet\">\n<script src=\"https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js\"></script>\n<script src=\"https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js\"></script>\n<script src=\"https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js\"></script>\n<style>\n:root{--bg:#0f1226;--pn:#171c3a;--ln:#283060;--tx:#e8ebff;--mu:#9aa3d6;--ac:#7cc4ff;--am:#ffb86b;--ok:#7be0a8;--bad:#ff7a8a}\n*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--tx);font:15px/1.55 Manrope,system-ui,sans-serif;display:flex;min-height:100vh}\naside{width:260px;padding:28px 20px;border-right:1px solid var(--ln);display:flex;flex-direction:column;gap:8px;position:sticky;top:0;height:100vh}\nh1{font-size:20px;font-weight:800;margin:0 0 4px;line-height:1.2}.sub{color:var(--mu);font-size:13px;margin-bottom:22px}\n.nav button{all:unset;display:block;width:100%;padding:12px 14px;border-radius:12px;cursor:pointer;color:var(--mu);font-weight:600;margin-bottom:6px;box-sizing:border-box}\n.nav button:hover{background:#1d2347}.nav button.on{background:var(--ac);color:#0c1230}\n#gpu{margin-top:auto;font-size:12px;color:var(--mu);border:1px solid var(--ln);border-radius:10px;padding:10px}\nmain{flex:1;padding:28px 32px;min-width:0}.view{display:none}.view.on{display:block}\nh2{font-size:26px;margin:0 0 4px;font-weight:800}.lead{color:var(--mu);margin:0 0 20px;max-width:75ch}\n.grid1{display:grid;grid-template-columns:minmax(0,1.5fr) minmax(300px,1fr);gap:18px}\n.card{background:var(--pn);border:1px solid var(--ln);border-radius:16px;padding:16px}\n.stage{height:480px;padding:0;overflow:hidden;position:relative}.stage canvas{width:100%;height:100%;display:block}\n#hud{position:absolute;left:14px;bottom:12px;right:14px;display:grid;grid-template-columns:1fr 1fr;gap:10px;font-size:12px;pointer-events:none}\n.bar{height:8px;background:#0c1030;border-radius:6px;overflow:hidden;margin-top:4px}.bar i{display:block;height:100%;width:0;border-radius:6px}\n.rs{display:flex;flex-direction:column;gap:8px}.rs button{all:unset;cursor:pointer;padding:11px 14px;border:1px solid var(--ln);border-radius:12px;font-weight:600;background:#121733}\n.rs button:hover{border-color:var(--ac)}.rs button.on{border-color:var(--am);background:#241f33;color:var(--am)}\n#why h3{margin:0 0 6px;font-size:18px}#why p{margin:0 0 10px;color:#cfd5ff}.tag{display:inline-block;font-size:12px;padding:3px 9px;border-radius:20px;border:1px solid var(--ln);color:var(--mu);margin-right:6px}\n.btn{all:unset;cursor:pointer;background:var(--am);color:#2a1700;font-weight:800;padding:12px 20px;border-radius:12px;text-align:center}.btn:disabled{opacity:.4;cursor:default}\n.ctl{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px;align-items:end}label{font-size:12px;color:var(--mu);display:block;margin-bottom:4px}\nselect,input[type=range]{width:100%;background:#0f1330;color:var(--tx);border:1px solid var(--ln);border-radius:10px;padding:9px}\n.drop{border:2px dashed var(--ln);border-radius:14px;padding:22px;text-align:center;color:var(--mu);cursor:pointer}.drop.h{border-color:var(--ac)}\n.pv{position:relative;margin-top:12px}.pv video,.pv img{width:100%;border-radius:12px;display:block;background:#000;max-height:420px}.pv canvas{position:absolute;inset:0;width:100%;height:100%;pointer-events:none}\n.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(130px,1fr));gap:12px;margin:18px 0}.kpi b{display:block;font-size:30px;font-weight:800}.kpi span{color:var(--mu);font-size:12px}\n.two{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:18px;margin-bottom:18px}\ntable{width:100%;border-collapse:collapse;font-size:13px}td,th{padding:7px 8px;border-bottom:1px solid var(--ln);text-align:left}th{color:var(--mu);font-weight:600}\n.chk label{display:inline-flex;gap:5px;margin-right:12px;color:var(--tx);font-size:13px}.prog{height:10px;background:#0c1030;border-radius:8px;overflow:hidden;margin-top:12px}.prog i{display:block;height:100%;width:0;background:var(--ac);transition:width .4s}\na.dl{color:var(--ac);font-weight:600;margin-right:16px}.ctrlbar{margin-top:12px;display:flex;flex-wrap:wrap;gap:10px 16px;align-items:center;font-size:13px}.ctrlbar .btn{padding:8px 14px;font-size:13px}.ctrlbar .sec,.btn.sec{background:#283060;color:var(--tx)}.ctrlbar input[type=range]{width:140px;padding:0;border:0;background:none;accent-color:var(--am)}.ctrlbar label{margin:0;color:var(--tx)}.tabs{display:flex;gap:8px;margin:16px 0 6px}.tabs button{all:unset;cursor:pointer;padding:9px 16px;border-radius:10px;border:1px solid var(--ln);font-weight:600;color:var(--mu)}.tabs button.on{background:var(--ac);color:#0c1230;border-color:var(--ac)}\n@media(max-width:900px){body{flex-direction:column}aside{width:auto;height:auto;position:static}.grid1{grid-template-columns:1fr}main{padding:18px}}\n</style></head><body>\n<aside><h1>Tracking 2D y trayectorias</h1><div class=\"sub\">Etapa 02 – Aplicaciones y conceptos</div>\n<div class=\"nav\"><button class=\"on\" data-v=\"v1\">Por qué la odometría visual es más difícil</button><button data-v=\"v2\">Tracking 2D con YOLO</button></div>\n<div id=\"gpu\">Conectando con la GPU…</div></aside>\n<main>\n<section class=\"view on\" id=\"v1\"><h2>Odometría visual 3D frente a tracking 2D</h2>\n<p class=\"lead\">Elige una razón. La escena 3D simula el proceso y compara la carga computacional de cada enfoque.</p>\n<div class=\"grid1\"><div><div class=\"card stage\"><canvas id=\"cv\"></canvas>\n<div id=\"hud\"><div><span>Tracking 2D (YOLO + Kalman)</span><div class=\"bar\"><i id=\"b2\" style=\"background:var(--ok)\"></i></div></div><div><span>Odometría visual / Pose 3D</span><div class=\"bar\"><i id=\"b3\" style=\"background:var(--am)\"></i></div></div><div id=\"mt\" style=\"grid-column:1/3;color:var(--ac);font-weight:600\"></div></div></div><div class=\"card ctrlbar\"><button class=\"btn\" id=\"pp\">Pausar</button><button class=\"btn sec\" id=\"rst\">Reiniciar</button><span>Velocidad <b id=\"sv\">1.00x</b></span><input type=\"range\" id=\"sp\" min=\"0.1\" max=\"2\" step=\"0.05\" value=\"1\"><span>Línea de tiempo</span><input type=\"range\" id=\"tl\" min=\"0\" max=\"1000\" value=\"0\" style=\"width:200px\"><label><input type=\"checkbox\" id=\"lp\" checked> Repetir en bucle</label><label><input type=\"checkbox\" id=\"ar\"> Rotación automática</label><button class=\"btn sec\" id=\"vr\">Restablecer vista</button><div style=\"color:var(--mu);width:100%;font-size:12px\">Arrastra para rotar, usa la rueda para acercar o alejar y el clic derecho para desplazar. En el móvil, un dedo rota y dos dedos acercan.</div></div></div>\n<div><div class=\"rs card\" id=\"rs\"></div><div class=\"card\" id=\"why\" style=\"margin-top:14px\"></div></div></div></section>\n\n<section class=\"view\" id=\"v2\"><h2>Tracking 2D: conteo y trayectorias</h2>\n<p class=\"lead\">Sube un video, ubica la línea virtual y analiza qué objetos la cruzan. El procesamiento se ejecuta en el servidor con GPU, si está disponible, o CPU.</p>\n<div class=\"card\"><div class=\"ctl\">\n<div><label>Algoritmo de seguimiento</label><select id=\"trk\"><option value=\"bytetrack\">YOLO + ByteTrack</option><option value=\"botsort\">YOLO + BoT-SORT</option></select></div>\n<div><label>Confianza mínima: <b id=\"cv_\">0.30</b></label><input type=\"range\" id=\"conf\" min=\"0.1\" max=\"0.8\" step=\"0.05\" value=\"0.3\"></div>\n<div><label>Orientación de la línea</label><select id=\"ori\"><option value=\"h\">Horizontal</option><option value=\"v\">Vertical</option></select></div>\n<div><label>Posición de la línea</label><input type=\"range\" id=\"pos\" min=\"5\" max=\"95\" value=\"50\"></div></div>\n<div class=\"chk\" style=\"margin-top:14px\"><label><input class=\"cls\" type=\"checkbox\" value=\"0\" checked>Persona</label><label><input class=\"cls\" type=\"checkbox\" value=\"1\">Bicicleta</label><label><input class=\"cls\" type=\"checkbox\" value=\"2\" checked>Auto</label><label><input class=\"cls\" type=\"checkbox\" value=\"3\" checked>Moto</label><label><input class=\"cls\" type=\"checkbox\" value=\"5\" checked>Bus</label><label><input class=\"cls\" type=\"checkbox\" value=\"7\" checked>Camión</label></div>\n<div class=\"tabs\"><button class=\"on\" data-m=\"mvid\">Archivo de video</button><button data-m=\"mlive\">Cámara en vivo</button></div><div id=\"mvid\"><div class=\"drop\" id=\"drop\" style=\"margin-top:14px\">Arrastra un video aquí o haz clic para elegirlo<input type=\"file\" id=\"file\" accept=\"video/*\" hidden></div>\n<div class=\"pv\" id=\"pvw\" style=\"display:none\"><video id=\"pv\" muted controls></video><canvas id=\"pc\"></canvas></div>\n<div style=\"display:flex;gap:14px;align-items:center;margin-top:14px\"><button class=\"btn\" id=\"go\" disabled>Analizar video</button><span id=\"st\" style=\"color:var(--mu)\"></span></div><div class=\"prog\"><i id=\"pg\"></i></div></div><div id=\"mlive\" style=\"display:none\"><div class=\"ctl\" style=\"margin-top:14px\"><div><label>Cámara (webcam o cámara externa)</label><select id=\"cam\"><option value=\"\">Cámara predeterminada</option></select></div><div style=\"display:flex;gap:10px\"><button class=\"btn\" id=\"lgo\">Iniciar cámara</button><button class=\"btn sec\" id=\"lst\" disabled>Detener y guardar</button></div></div><div id=\"lerr\" style=\"color:var(--bad);margin-top:8px\"></div><div class=\"pv\"><canvas id=\"lc\" style=\"position:static;height:auto;width:100%;border-radius:12px;background:#000;min-height:220px\"></canvas></div><div class=\"kpis\" id=\"lk\"></div><div class=\"two\"><div><b>Objetos activos en vivo</b><canvas id=\"lch\"></canvas></div><div><b>Resumen por ID</b><table id=\"ltb\"></table><p id=\"ldl\"></p></div></div><video id=\"lv\" playsinline muted style=\"display:none\"></video></div></div>\n<div id=\"out\" style=\"display:none\">\n<div class=\"kpis\" id=\"kp\"></div>\n<div class=\"two\"><div class=\"card\"><video id=\"rv\" controls style=\"width:100%;border-radius:12px\"></video></div>\n<div class=\"card\"><div style=\"display:flex;justify-content:space-between\"><b>Trayectorias</b><label style=\"margin:0\"><input type=\"checkbox\" id=\"hm\"> Mapa de calor</label></div><div class=\"pv\"><canvas id=\"tc\" style=\"position:static;height:auto;width:100%;border-radius:12px\"></canvas></div></div></div>\n<div class=\"two\"><div class=\"card\"><b>Objetos activos por instante</b><canvas id=\"c1\"></canvas></div><div class=\"card\"><b>Cruces de la línea por clase</b><canvas id=\"c2\"></canvas></div></div>\n<div class=\"card\"><b>Objetos con mayor recorrido</b><table id=\"tb\"></table><p id=\"dls\" style=\"margin-bottom:0\"></p></div></div></section></main>\n<script>\nconst $=s=>document.querySelector(s),$$=s=>[...document.querySelectorAll(s)];\n$$('.nav button').forEach(b=>b.onclick=()=>{$$('.nav button').forEach(x=>x.classList.toggle('on',x==b));$$('.view').forEach(v=>v.classList.toggle('on',v.id==b.dataset.v));resize()});\nconst gi=()=>fetch('/api/info').then(r=>r.json()).then(i=>$('#gpu').textContent=i.gpu+' – '+i.running+'/'+i.slots+' análisis activos, '+i.queue+' en cola').catch(()=>{});gi();setInterval(gi,5000);\n/* ---------- Ventana 1: escenas 3D ---------- */\nlet R,S,C,G,AL,OC,cur=null,DUR=8,tt=0,playing=true,spd=1,LT=performance.now();\nconst M=(c)=>new THREE.MeshStandardMaterial({color:c,roughness:.4,metalness:.2});\nconst LN=(pts,c)=>{const l=new THREE.Line(new THREE.BufferGeometry().setFromPoints(pts),new THREE.LineBasicMaterial({color:c,transparent:true}));G.add(l);return l};\nconst sph=(r,c,p)=>{const m=new THREE.Mesh(new THREE.SphereGeometry(r,14,14),M(c));m.position.set(...p);G.add(m);return m};\nconst box=(w,h,d,c,p)=>{const m=new THREE.Mesh(new THREE.BoxGeometry(w,h,d),M(c));m.position.set(...p);G.add(m);return m};\nfunction lab(s,p,sc=3.2){const c=document.createElement('canvas');c.width=512;c.height=96;const x=c.getContext('2d');x.font='600 42px Manrope,sans-serif';x.fillStyle='#e8ebff';x.textAlign='center';x.fillText(s,256,62);const sp=new THREE.Sprite(new THREE.SpriteMaterial({map:new THREE.CanvasTexture(c),transparent:true}));sp.position.set(...p);sp.scale.set(sc,sc*96/512,1);G.add(sp)}\nconst cone=c=>{const m=new THREE.Mesh(new THREE.ConeGeometry(.5,1.2,4),M(c));m.rotation.z=-Math.PI/2;return m};\nconst rnd=(()=>{let s=7;return()=>(s=(s*16807)%2147483647)/2147483647})();\nconst ez=t=>t*t*(3-2*t);\nconst SC=[\n{n:'Seis grados de libertad',l:[8,38],\n why:'En 2D basta con ubicar cada objeto en la imagen: dos coordenadas (x, y). En odometría visual hay que estimar la pose completa de la cámara: tres traslaciones y tres rotaciones. Un error pequeño de rotación se amplifica con la distancia y desplaza todo el mapa estimado.',\n run(){box(5,.05,4,0x232a55,[-4.5,0,0]);const a=sph(.3,0x7cc4ff,[0,.3,0]);lab('2D: 2 parámetros',[-4.5,2.6,0]);lab('3D: 6 parámetros',[4.5,3.4,0]);\n  const cam=new THREE.Group();cam.add(cone(0xffb86b),new THREE.AxesHelper(1.6));G.add(cam);\n  return t=>{const u=t*6.28;a.position.set(-4.5+Math.cos(u)*1.8,.3,Math.sin(u)*1.4);cam.position.set(4.5+Math.cos(u)*1.8,1.5+Math.sin(u*2)*.8,Math.sin(u)*1.4);cam.rotation.set(t*5,t*7,t*3);return 'Parámetros a estimar por cuadro: '+(t<.1?'…':'2 frente a 6')}}},\n{n:'Ambigüedad de escala',l:[10,45],\n why:'Con una sola cámara, un objeto pequeño y cercano produce la misma imagen que uno grande y lejano. La escala del movimiento no se puede observar sin estéreo, IMU, sensor de profundidad o conocimiento previo. En tracking 2D no hace falta: se trabaja directamente en píxeles.',\n run(){const c=cone(0xffb86b);c.position.set(-6,0,0);G.add(c);LN([new THREE.Vector3(-6,0,0),new THREE.Vector3(5,2.2,0)],0x7cc4ff);LN([new THREE.Vector3(-6,0,0),new THREE.Vector3(5,-2.2,0)],0x7cc4ff);\n  box(.03,.8,.8,0x7cc4ff,[-4.5,0,0]);lab('Misma imagen',[-4.5,1.4,0],2.4);const o=box(1,1,1,0xff7a8a,[0,0,0]);\n  return t=>{const d=3+8*(.5+.5*Math.sin(t*6.28-1.57));o.position.x=-6+d;o.scale.setScalar(.4*d);return 'Distancia '+d.toFixed(1)+' m, tamaño '+(.8*d).toFixed(1)+' m: indistinguibles en la imagen'}}},\n{n:'Deriva acumulada',l:[6,70],\n why:'La odometría suma movimientos relativos cuadro a cuadro, y cada pequeño error se acumula sin límite. Para frenar la deriva se necesitan cierres de bucle y optimización global (pose graph, bundle adjustment). En tracking 2D, un error en un cuadro se corrige con la detección del cuadro siguiente.',\n run(){const N=200,gt=[],es=[];let ex=0,ey=0;for(let i=0;i<N;i++){const a=i/N*6.28;ex+=Math.sin(i*1.7)*.02+.013;ey+=Math.cos(i*2.3)*.02+.009;gt.push(new THREE.Vector3(Math.cos(a)*4,0,Math.sin(a)*4));es.push(new THREE.Vector3(Math.cos(a)*4+ex,0,Math.sin(a)*4+ey))}\n  const g=LN(gt,0x7be0a8),e=LN(es,0xff7a8a);lab('Trayectoria real',[-5,.5,5.2],3);lab('Trayectoria estimada',[5,.5,5.2],3);const h=sph(.2,0xffb86b,[4,0,0]);\n  return t=>{const k=Math.max(2,Math.floor(t*N));g.geometry.setDrawRange(0,k);e.geometry.setDrawRange(0,k);h.position.copy(es[k-1]);return 'Error de posición acumulado: '+gt[k-1].distanceTo(es[k-1]).toFixed(2)+' m'}}},\n{n:'Correspondencias y outliers',l:[12,62],\n why:'Hay que detectar puntos característicos (ORB, SIFT, SuperPoint), emparejarlos entre cuadros y descartar los falsos con RANSAC sobre la geometría epipolar. Unos pocos emparejamientos erróneos bastan para arruinar la pose. El tracking 2D solo compara cajas por IoU y apariencia.',\n run(){box(3.6,4.2,.05,0x232a55,[-4,0,0]);box(3.6,4.2,.05,0x232a55,[4,0,0]);lab('Cuadro t',[-4,2.7,0],2.4);lab('Cuadro t+1',[4,2.7,0],2.4);\n  const N=36,ls=[],bad=[];for(let i=0;i<N;i++){const x=(rnd()-.5)*3,y=(rnd()-.5)*3.6,o=rnd()<.3;bad.push(o);const q=o?[(rnd()-.5)*3,(rnd()-.5)*3.6]:[x+.2,y+.1];ls.push(LN([new THREE.Vector3(-4+x,y,.1),new THREE.Vector3(4+q[0],q[1],.1)],0x7cc4ff));ls[i].visible=false}\n  return t=>{const v=Math.floor(Math.min(1,t*3)*N);let ok=0;ls.forEach((l,i)=>{l.visible=i<v;const o=bad[i];l.material.color.set(t>.45?(o?0xff7a8a:0x7be0a8):0x7cc4ff);l.material.opacity=(t>.75&&o)?Math.max(0,1-(t-.75)*8):1;if(!o)ok++});return t<.45?'Emparejando puntos…':t<.75?'RANSAC: iterando hipótesis':'Inliers conservados: '+ok+' de '+N}}},\n{n:'Cámara en movimiento',l:[9,55],\n why:'En seguridad la cámara está fija: el fondo no cambia y todo lo que se mueve es un objeto. En odometría la cámara se mueve, así que todo el fondo se desplaza (ego-motion) y hay que separar ese flujo del movimiento propio de personas y vehículos.',\n run(){const n=500,pos=new Float32Array(n*3);for(let i=0;i<n;i++){pos[i*3]=(rnd()-.5)*16;pos[i*3+1]=(rnd()-.5)*6;pos[i*3+2]=(rnd()-.5)*16}const z0=Float32Array.from({length:n},(_,i)=>pos[i*3+2]);\n  const g=new THREE.BufferGeometry();g.setAttribute('position',new THREE.BufferAttribute(pos,3));G.add(new THREE.Points(g,new THREE.PointsMaterial({color:0x9aa3d6,size:.09})));\n  const cs=[0xffb86b,0xff7a8a,0x7be0a8].map((c,i)=>box(.8,.8,.8,c,[0,0,0]));\n  return t=>{const ego=t>.45;for(let i=0;i<n;i++)pos[i*3+2]=((z0[i]+8+(ego?(t-.45)*40:0))%16+16)%16-8;g.attributes.position.needsUpdate=true;\n   cs.forEach((c,i)=>c.position.set(Math.sin(t*9+i*2)*5,.5+i*.8-1,Math.cos(t*7+i)*3));return ego?'Cámara móvil: todo el fondo se mueve':'Cámara fija: solo se mueven los objetos'}}},\n{n:'Carga computacional',l:[14,95],\n why:'El tracking 2D ejecuta un detector, un filtro de Kalman y una asociación húngara: unas decenas de milisegundos en GPU. La odometría añade extracción de puntos, emparejamiento, RANSAC, PnP o triangulación, bundle adjustment local, mapa y cierre de bucle: optimización no lineal con miles de variables que debe cumplirse en tiempo real.',\n run(){const a=['Detector','Kalman','Asociación'],b=['Features','Matching','RANSAC','PnP','Bundle Adj.','Mapa'],ta=[1.4,.7,.5],tb=[1.4,1.2,2.8,1.6,4.2,2.6],bs=[];\n  a.forEach((s,i)=>{bs.push([box(1.4,1,1.4,0x7be0a8,[-2+i*2,0,-3]),ta[i],i]);lab(s,[-2+i*2,4.6,-3],2.4)});b.forEach((s,i)=>{bs.push([box(1.4,1,1.4,0xffb86b,[-5+i*2,0,3]),tb[i],i+3]);lab(s,[-5+i*2,5.2,3],2.4)});\n  return t=>{const k=ez(Math.min(1,t*2));bs.forEach(([m,h,i])=>{const H=Math.max(.05,h*k*(.88+.12*Math.sin(t*50+i)));m.scale.y=H;m.position.y=H/2});return 'Tiempo por cuadro estimado: 2D '+(12*k).toFixed(0)+' ms, 3D '+(85*k).toFixed(0)+' ms'}}},\n{n:'Condiciones adversas',l:[18,85],\n why:'La odometría depende de textura e iluminación estables. La poca luz, el desenfoque por movimiento, las paredes lisas, los reflejos y el rolling shutter hacen perder puntos y la estimación se rompe. Los detectores profundos son más robustos porque aprenden la apariencia del objeto, no la geometría del entorno.',\n run(){const P=[];for(let i=0;i<60;i++){P.push([sph(.14,0x7be0a8,[-4.5+(i%10),Math.floor(i/10)-2,0]),rnd()*.85])}box(10,6,.05,0x1b2145,[0,0,-.2]);\n  return t=>{const L=1-t*.95;AL.intensity=.15+.55*L;let n=0;P.forEach(([m,th],i)=>{const ok=L>th;if(ok)n++;m.material.color.set(ok?0x7be0a8:0xff7a8a);m.scale.setScalar(ok?1:.4);m.position.x=-4.5+(i%10)+Math.sin(i*12.9+t*80)*.05*t});return 'Puntos rastreados: '+n+' de 60'+(n<15?'. Seguimiento perdido':'')}}}\n];\nfunction init3(){const el=$('#cv');R=new THREE.WebGLRenderer({canvas:el,antialias:true});S=new THREE.Scene();S.background=new THREE.Color(0x101426);C=new THREE.PerspectiveCamera(50,1,.1,100);C.position.set(0,6,13);OC=THREE.OrbitControls?new THREE.OrbitControls(C,el):{update(){},target:new THREE.Vector3()};OC.target.set(0,1,0);OC.enableDamping=true;OC.dampingFactor=.08;OC.minDistance=3;OC.maxDistance=45;OC.autoRotateSpeed=1.2;\n AL=new THREE.AmbientLight(0xffffff,.7);S.add(AL);const d=new THREE.DirectionalLight(0xffffff,.8);d.position.set(5,8,5);S.add(d);G=new THREE.Group();S.add(G);\n const rs=$('#rs');SC.forEach((s,i)=>{const b=document.createElement('button');b.textContent=s.n;b.onclick=()=>play(i);rs.appendChild(b)});resize();bind3();play(0);loop()}\nfunction resize(){if(!R)return;const p=$('#cv').parentElement,w=p.clientWidth,h=p.clientHeight;if(!w)return;R.setSize(w,h,false);C.aspect=w/h;C.updateProjectionMatrix()}\naddEventListener('resize',resize);\nfunction bind3(){$('#pp').onclick=()=>{if(!playing&&tt>=1)tt=0;playing=!playing;$('#pp').textContent=playing?'Pausar':'Reproducir'};$('#rst').onclick=()=>{tt=0;playing=true;$('#pp').textContent='Pausar'};$('#sp').oninput=e=>{spd=+e.target.value;$('#sv').textContent=spd.toFixed(2)+'x'};$('#tl').oninput=e=>{tt=e.target.value/1000;playing=false;$('#pp').textContent='Reproducir'};$('#ar').onchange=e=>OC.autoRotate=e.target.checked;$('#vr').onclick=()=>{C.position.set(0,6,13);OC.target.set(0,1,0);OC.update()}}\nfunction play(i){[...$('#rs').children].forEach((b,k)=>b.classList.toggle('on',k==i));while(G.children.length)G.remove(G.children[0]);AL.intensity=.7;G.add(new THREE.GridHelper(20,20,0x2a3160,0x1d2347));\n const s=SC[i];cur={tick:s.run(),l:s.l};tt=0;playing=true;$('#pp').textContent='Pausar';$('#why').innerHTML='<h3>'+s.n+'</h3><p>'+s.why+'</p><span class=\"tag\">2D: carga '+s.l[0]+'%</span><span class=\"tag\">3D: carga '+s.l[1]+'%</span><p style=\"margin-top:12px\"><button class=\"btn\" onclick=\"play('+i+')\">Repetir animación</button></p>'}\nfunction loop(){requestAnimationFrame(loop);const now=performance.now(),dt=Math.min((now-LT)/1000,.1);LT=now;if(cur){if(playing){tt=Math.min(1,tt+dt*spd/DUR);if(tt>=1){if($('#lp').checked)tt=0;else{playing=false;$('#pp').textContent='Reproducir'}}}const t=tt;$('#tl').value=t*1000;$('#mt').textContent=cur.tick(t);const k=ez(Math.min(1,t*2.2));$('#b2').style.width=cur.l[0]*k+'%';$('#b3').style.width=cur.l[1]*k+'%'}\n OC.update();R.render(S,C)}\ninit3();\n/* ---------- Ventana 2: tracking ---------- */\nlet FILE=null,RES=null,JID=null;\n$('#conf').oninput=e=>$('#cv_').textContent=(+e.target.value).toFixed(2);\n$('#drop').onclick=()=>$('#file').click();$('#file').onchange=e=>e.target.files[0]&&setFile(e.target.files[0]);\n$('#drop').ondragover=e=>{e.preventDefault();$('#drop').classList.add('h')};$('#drop').ondragleave=()=>$('#drop').classList.remove('h');\n$('#drop').ondrop=e=>{e.preventDefault();$('#drop').classList.remove('h');e.dataTransfer.files[0]&&setFile(e.dataTransfer.files[0])};\nfunction setFile(f){FILE=f;$('#pvw').style.display='block';const v=$('#pv');v.src=URL.createObjectURL(f);v.onloadeddata=drawLine;$('#go').disabled=false;$('#drop').firstChild.textContent=f.name}\nconst lineP=()=>{const p=$('#pos').value/100;return $('#ori').value=='h'?[0,p,1,p]:[p,0,p,1]};\nfunction drawLine(){const v=$('#pv'),c=$('#pc');c.width=v.clientWidth;c.height=v.clientHeight;const x=c.getContext('2d'),l=lineP();x.setLineDash([10,6]);x.strokeStyle='#ffb86b';x.lineWidth=3;x.beginPath();x.moveTo(l[0]*c.width,l[1]*c.height);x.lineTo(l[2]*c.width,l[3]*c.height);x.stroke()}\n$('#pos').oninput=$('#ori').onchange=drawLine;\n$('#go').onclick=async()=>{const fd=new FormData();fd.append('file',FILE);fd.append('tracker',$('#trk').value);fd.append('conf',$('#conf').value);fd.append('line',JSON.stringify(lineP()));fd.append('classes',$$('.cls:checked').map(c=>c.value).join(','));\n $('#go').disabled=true;$('#st').textContent='Subiendo video…';$('#out').style.display='none';\n try{const r=await(await fetch('/api/track',{method:'POST',body:fd})).json();JID=r.id;poll()}catch(e){$('#st').textContent='No se pudo subir el video. Revisa tu conexión e inténtalo de nuevo.';$('#go').disabled=false}};\nasync function poll(){const j=await(await fetch('/api/job/'+JID)).json();$('#pg').style.width=(j.p||0)*100+'%';\n if(j.state=='done'){$('#st').textContent='Análisis completo';$('#go').disabled=false;show(j.res)}\n else if(j.state=='error'){$('#st').textContent='Error: '+j.err;$('#go').disabled=false}\n else{$('#st').textContent=j.state=='queued'?'En cola, esperando un espacio de procesamiento…':'Procesando en el servidor: '+Math.round((j.p||0)*100)+'%';setTimeout(poll,800)}}\nlet ch=[];\nfunction show(r){RES=r;$('#out').style.display='block';$('#rv').src='/files/'+JID+'/video.mp4';\n const k=[['Entran',r.cin],['Salen',r.cout],['Objetos únicos (IDs)',r.ids],['Cuadros analizados',r.frames],['FPS de inferencia',r.inf_fps],['FPS del video',r.fps],['Tamaño del video (MB)',r.mb]];\n $('#kp').innerHTML=k.map(x=>'<div class=\"card kpi\"><b>'+x[1]+'</b><span>'+x[0]+'</span></div>').join('');\n ch.forEach(c=>c.destroy());const o={plugins:{legend:{labels:{color:'#9aa3d6'}}},scales:{x:{ticks:{color:'#9aa3d6'},grid:{color:'#283060'}},y:{ticks:{color:'#9aa3d6'},grid:{color:'#283060'}}}};\n ch=[new Chart($('#c1'),{type:'line',data:{labels:r.series.map((_,i)=>((i*r.st)/r.fps).toFixed(1)+' s'),datasets:[{label:'Objetos activos',data:r.series,borderColor:'#7cc4ff',pointRadius:0,tension:.3,fill:true,backgroundColor:'#7cc4ff22'}]},options:o}),\n new Chart($('#c2'),{type:'bar',data:{labels:Object.keys(r.pc),datasets:[{label:'Entran',data:Object.values(r.pc).map(v=>v[0]),backgroundColor:'#7be0a8'},{label:'Salen',data:Object.values(r.pc).map(v=>v[1]),backgroundColor:'#ff7a8a'}]},options:o})];\n $('#tb').innerHTML='<tr><th>ID</th><th>Clase</th><th>Duración (s)</th><th>Recorrido (px)</th><th>Velocidad media (px/s)</th></tr>'+r.rows.slice(0,15).map(x=>'<tr><td>#'+x.id+'</td><td>'+x.cls+'</td><td>'+x.dur+'</td><td>'+x.dist+'</td><td>'+x.vel+'</td></tr>').join('');\n $('#dls').innerHTML=['trayectorias.csv','resultados.json'].map(n=>'<a class=\"dl\" href=\"/files/'+JID+'/'+n+'\" download>Descargar '+n+'</a>').join('');drawT()}\n$('#hm').onchange=()=>drawT();\nfunction drawT(){const r=RES,im=new Image();im.onload=()=>{const c=$('#tc');c.width=r.w;c.height=r.h;const x=c.getContext('2d');x.drawImage(im,0,0);x.fillStyle='#0f1226aa';x.fillRect(0,0,r.w,r.h);\n if($('#hm').checked){const mx=Math.max(...r.heat.flat(),1),cw=r.w/32,chh=r.h/18;r.heat.forEach((row,j)=>row.forEach((v,i)=>{if(v){x.fillStyle='rgba(255,184,107,'+(.15+.8*v/mx)+')';x.fillRect(i*cw,j*chh,cw,chh)}}))}\n else r.tr.forEach(t=>{x.strokeStyle='hsl('+(t.id*47%360)+',90%,65%)';x.lineWidth=Math.max(2,r.w/500);x.beginPath();t.p.forEach((q,i)=>i?x.lineTo(q[0],q[1]):x.moveTo(q[0],q[1]));x.stroke()});\n const l=r.line;x.strokeStyle='#ffb86b';x.lineWidth=Math.max(3,r.w/400);x.setLineDash([16,10]);x.beginPath();x.moveTo(l[0]*r.w,l[1]*r.h);x.lineTo(l[2]*r.w,l[3]*r.h);x.stroke()};im.src='/files/'+JID+'/poster.jpg'}\n/* ---------- Cámara en vivo ---------- */\n$$('.tabs button').forEach(b=>b.onclick=()=>{$$('.tabs button').forEach(x=>x.classList.toggle('on',x==b));$('#mvid').style.display=b.dataset.m=='mvid'?'block':'none';$('#mlive').style.display=b.dataset.m=='mlive'?'block':'none';$('#out').style.display=(b.dataset.m=='mvid'&&RES)?'block':'none';if(b.dataset.m=='mlive')listCams()});\nlet WS=null,STRM=null,LAST=null,TRK={},ROWS=[],LF=0,LCH=null,LT0=0,LP=0,LFPS=0,RUN=false;const OFF=document.createElement('canvas');\nconst cfg=()=>({tracker:$('#trk').value,conf:+$('#conf').value,line:lineP(),classes:$$('.cls:checked').map(c=>+c.value)});\n['conf','pos','ori'].forEach(id=>$('#'+id).addEventListener('input',()=>{if(WS&&WS.readyState==1)WS.send(JSON.stringify(cfg()))}));\nasync function listCams(){try{const d=(await navigator.mediaDevices.enumerateDevices()).filter(x=>x.kind=='videoinput'),s=$('#cam'),v=s.value;s.innerHTML='<option value=\"\">Cámara predeterminada</option>'+d.map((x,i)=>'<option value=\"'+x.deviceId+'\">'+(x.label||'Cámara '+(i+1))+'</option>').join('');s.value=v}catch(e){}}\n$('#lgo').onclick=async()=>{$('#lerr').textContent='';\n try{STRM=await navigator.mediaDevices.getUserMedia({video:{deviceId:$('#cam').value?{exact:$('#cam').value}:undefined,width:{ideal:1280},height:{ideal:720}},audio:false})}catch(e){$('#lerr').textContent='No se pudo abrir la cámara. Permite el acceso en el navegador o elige otra cámara.';return}\n const v=$('#lv');v.srcObject=STRM;await v.play();listCams();\n TRK={};ROWS=[];LF=0;LAST=null;LP=0;LFPS=0;RUN=true;LT0=performance.now();$('#lgo').disabled=true;$('#lst').disabled=false;$('#ltb').innerHTML='';$('#ldl').innerHTML='';\n if(LCH)LCH.destroy();LCH=new Chart($('#lch'),{type:'line',data:{labels:[],datasets:[{label:'Objetos activos',data:[],borderColor:'#7cc4ff',pointRadius:0,tension:.3,fill:true,backgroundColor:'#7cc4ff22'}]},options:{animation:false,plugins:{legend:{display:false}},scales:{x:{ticks:{color:'#9aa3d6',maxTicksLimit:6},grid:{color:'#283060'}},y:{beginAtZero:true,ticks:{color:'#9aa3d6',precision:0},grid:{color:'#283060'}}}}});\n WS=new WebSocket((location.protocol=='https:'?'wss://':'ws://')+location.host+'/ws/live');\n WS.onopen=()=>{WS.send(JSON.stringify(cfg()));sendF()};\n WS.onmessage=e=>{const r=JSON.parse(e.data);if(r.error){$('#lerr').textContent=r.error;stopLive(false);return}onRes(r);sendF()};\n WS.onclose=()=>{if(RUN)stopLive(false)};drawL()};\n$('#lst').onclick=()=>stopLive(true);\nfunction sendF(){if(!RUN)return;const v=$('#lv');if(WS.readyState!=1||!v.videoWidth)return setTimeout(sendF,50);const w=Math.min(640,v.videoWidth);OFF.width=w;OFF.height=Math.round(w*v.videoHeight/v.videoWidth);OFF.getContext('2d').drawImage(v,0,0,OFF.width,OFF.height);OFF.toBlob(b=>{if(RUN&&WS.readyState==1)WS.send(b)},'image/jpeg',.7)}\nfunction onRes(r){LAST=r;LF++;const v=$('#lv'),s=v.videoWidth/r.w,n=performance.now(),t=(n-LT0)/1000;LFPS=LP?LFPS*.85+.15*(1000/(n-LP)):1;LP=n;\n r.b.forEach(b=>{const cx=(b[0]+b[2])/2*s,cy=(b[1]+b[3])/2*s;(TRK[b[4]]=TRK[b[4]]||{n:b[6],p:[]}).p.push([cx,cy]);ROWS.push([b[4],b[6],LF,t.toFixed(3),cx.toFixed(1),cy.toFixed(1)])});\n $('#lk').innerHTML=[['Entran',r.cin],['Salen',r.cout],['Activos',r.b.length],['IDs únicos',Object.keys(TRK).length],['FPS de extremo a extremo',LFPS.toFixed(1)],['Latencia de la GPU (ms)',r.ms]].map(x=>'<div class=\"card kpi\"><b>'+x[1]+'</b><span>'+x[0]+'</span></div>').join('');\n if(LF%3==0){const d=LCH.data;d.labels.push(t.toFixed(0)+' s');d.datasets[0].data.push(r.b.length);if(d.labels.length>150){d.labels.shift();d.datasets[0].data.shift()}LCH.update('none')}}\nfunction drawL(full){const v=$('#lv'),c=$('#lc');if(!v.videoWidth){if(RUN)requestAnimationFrame(()=>drawL());return}\n c.width=v.videoWidth;c.height=v.videoHeight;const x=c.getContext('2d'),W=c.width,H=c.height,lw=Math.max(2,W/400);x.drawImage(v,0,0,W,H);\n const l=lineP();x.strokeStyle='#ffb86b';x.lineWidth=lw*1.5;x.setLineDash([14,8]);x.beginPath();x.moveTo(l[0]*W,l[1]*H);x.lineTo(l[2]*W,l[3]*H);x.stroke();x.setLineDash([]);\n for(const id in TRK){const p=full?TRK[id].p:TRK[id].p.slice(-60);x.strokeStyle='hsl('+(id*47%360)+',90%,65%)';x.lineWidth=lw;x.beginPath();p.forEach((q,i)=>i?x.lineTo(q[0],q[1]):x.moveTo(q[0],q[1]));x.stroke()}\n if(LAST&&!full){const s=W/LAST.w;x.font='600 '+Math.max(14,W/50)+'px Manrope,sans-serif';LAST.b.forEach(b=>{const col='hsl('+(b[4]*47%360)+',90%,65%)';x.strokeStyle=col;x.fillStyle=col;x.lineWidth=lw*1.2;x.strokeRect(b[0]*s,b[1]*s,(b[2]-b[0])*s,(b[3]-b[1])*s);x.fillText('#'+b[4]+' '+b[6],b[0]*s,Math.max(14,b[1]*s-6))})}\n if(LAST){x.fillStyle='#0f1226cc';x.fillRect(0,0,W/4,H/6);x.font='700 '+Math.max(16,W/40)+'px Manrope,sans-serif';x.fillStyle='#7be0a8';x.fillText('Entran: '+LAST.cin,12,H/20);x.fillStyle='#ff7a8a';x.fillText('Salen: '+LAST.cout,12,H/10)}\n if(RUN&&!full)requestAnimationFrame(()=>drawL())}\nfunction stopLive(save){if(!RUN)return;RUN=false;if(save)drawL(true);try{WS.close()}catch(e){}STRM&&STRM.getTracks().forEach(t=>t.stop());$('#lgo').disabled=false;$('#lst').disabled=true;if(save)summary()}\nfunction summary(){const rows=Object.entries(TRK).map(([id,t])=>{let d=0;for(let i=1;i<t.p.length;i++)d+=Math.hypot(t.p[i][0]-t.p[i-1][0],t.p[i][1]-t.p[i-1][1]);return{id,n:t.n,f:t.p.length,d:Math.round(d)}}).sort((a,b)=>b.d-a.d).slice(0,15);\n $('#ltb').innerHTML='<tr><th>ID</th><th>Clase</th><th>Detecciones</th><th>Recorrido (px)</th></tr>'+rows.map(x=>'<tr><td>#'+x.id+'</td><td>'+x.n+'</td><td>'+x.f+'</td><td>'+x.d+'</td></tr>').join('');\n const csv='id,clase,frame,tiempo_s,cx,cy\\n'+ROWS.map(r=>r.join(',')).join('\\n');$('#ldl').innerHTML='<a class=\"dl\" download=\"trayectorias_en_vivo.csv\" href=\"'+URL.createObjectURL(new Blob([csv],{type:'text/csv'}))+'\">Descargar trayectorias_en_vivo.csv</a>'}\n</script></body></html>\"\"\"\n\ndef main():\n    import uvicorn\n    server = uvicorn.Server(uvicorn.Config(app, host=\"127.0.0.1\", port=8000, log_level=\"warning\"))\n    thread = threading.Thread(target=server.run, daemon=True)\n    tunnel = None\n    try:\n        print(\"Cargando\", NPOOL, \"instancias de\", WEIGHTS, \"en\", torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\", flush=True)\n        load_models()\n        thread.start()\n        deadline = time.monotonic() + 30\n        while not server.started:\n            if not thread.is_alive() or time.monotonic() > deadline:\n                raise RuntimeError(\"No se pudo iniciar el servidor en el puerto 8000.\")\n            time.sleep(0.1)\n        binary = Path(\"cloudflared\")\n        if not binary.is_file():\n            temporary = binary.with_suffix(\".download\")\n            try:\n                subprocess.run([\"wget\", \"-q\", \"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\", \"-O\", str(temporary)], check=True)\n                temporary.chmod(0o700)\n                temporary.replace(binary)\n            finally:\n                temporary.unlink(missing_ok=True)\n        tunnel = subprocess.Popen([str(binary.resolve()), \"tunnel\", \"--url\", \"http://127.0.0.1:8000\", \"--no-autoupdate\"], stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)\n        for line in tunnel.stderr:\n            match = re.search(r\"https://[-a-z0-9]+\\.trycloudflare\\.com\", line)\n            if match:\n                print(\"\\nLINK PÚBLICO:\", match.group(0), \"\\nDescarga los resultados antes de interrumpir esta celda.\\n\", flush=True)\n            elif \"error\" in line.lower() or \"failed\" in line.lower():\n                print(line.strip(), flush=True)\n        raise RuntimeError(\"El túnel terminó. Ejecuta de nuevo la celda para obtener otro enlace.\")\n    except KeyboardInterrupt:\n        print(\"\\nCerrando el servidor y el túnel…\", flush=True)\n    finally:\n        stopping.set()\n        if tunnel is not None:\n            tunnel.terminate()\n            try:\n                tunnel.wait(timeout=5)\n            except subprocess.TimeoutExpired:\n                tunnel.kill()\n                tunnel.wait()\n            if tunnel.stderr:\n                tunnel.stderr.close()\n        server.should_exit = True\n        if thread.ident is not None:\n            thread.join(timeout=5)\n        ex.shutdown(wait=True, cancel_futures=True)\n\n\nif __name__ == \"__main__\":\n    main()\n"
}
REQUIREMENTS = {
    "odometria_visual": "ultralytics\nlapx\nfastapi\nuvicorn[standard]\npython-multipart\nnumpy\nopencv-python\ntorch\n",
    "radar_trafico": "ultralytics>=8.4,<9\nlap\nfastapi\nuvicorn[standard]\npython-multipart\nav\nmatplotlib\nnumpy\nopencv-python\ntorch\npydantic>=2,<3\n",
    "tracking_2d": "ultralytics\nlapx\nfastapi\nuvicorn[standard]\npython-multipart\nnumpy\nopencv-python\ntorch\n"
}

In [ ]:
# @title 3. Instalar con uv y preparar la experiencia
INSTALLED_APP = None
APP.mkdir(parents=True, exist_ok=True)
(APP / "app.py").write_text(SOURCES[APP.name])
(APP / "requirements.txt").write_text(REQUIREMENTS[APP.name])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--system", "--python", sys.executable, "-r", str(APP / "requirements.txt")], check=True)
needed = [tool for tool in ("wget", "ffmpeg") if (tool == "wget" or APP.name == "tracking_2d") and not shutil.which(tool)]
if needed:
    if not shutil.which("apt-get") or os.geteuid() != 0:
        raise RuntimeError(f"Faltan herramientas del sistema: {', '.join(needed)}. Utiliza un entorno de Colab.")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", *needed], check=True)
subprocess.run([sys.executable, "-c", "import torch; print('Dispositivo:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"], check=True)
if APP.name == "radar_trafico":
    subprocess.run([sys.executable, str(APP / "app.py"), "--check"], check=True)
INSTALLED_APP = APP
print("Listo. Ejecuta la siguiente celda.")

## Abrir la aplicación

La siguiente celda permanece activa. Usa el enlace que aparezca en la salida.

Para finalizar, descarga los resultados desde la interfaz y pulsa **Detener** en Colab. El cierre espera hasta 15 segundos antes de forzar la salida; un análisis interrumpido puede quedar incompleto.

In [ ]:
# @title 4. Ejecutar la experiencia
def run_app(script, env):
    # Cada ejecución posee su grupo para cerrar también túnel y procesos hijos.
    with socket.socket() as probe:
        probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            probe.bind(("127.0.0.1", 8000))
        except OSError as exc:
            raise RuntimeError("El puerto 8000 está ocupado. Detén la ejecución anterior antes de continuar.") from exc
    process = subprocess.Popen(
        [sys.executable, "-u", str(script)], cwd=ROOT, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        bufsize=1, start_new_session=True,
    )
    interrupted = False
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)
    except KeyboardInterrupt:
        interrupted = True
        print("\nCerrando la experiencia…", flush=True)
    finally:
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            print("Se agotó el tiempo de cierre; se forzará la salida.", flush=True)
        finally:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
            process.stdout.close()
    if interrupted:
        print("Experiencia detenida. Ya puedes cambiar la selección.")

if INSTALLED_APP != APP:
    raise RuntimeError("Instala los requisitos de la experiencia seleccionada antes de ejecutarla.")
env = os.environ.copy()
env["NPOOL"] = "1"
env["JOBS_DIR"] = str(ROOT / ("traffic-monitor" if APP.name == "radar_trafico" else "jobs"))
run_app(APP / "app.py", env)

## Guardar resultados

- **Tracking:** `trayectorias.csv`, `resultados.json`; en webcam, `trayectorias_en_vivo.csv` al pulsar «Detener y guardar».
- **Radar:** video anotado, `vehicles.csv`, `heatmap.png` y `traffic-by-time.png`.
- **Odometría:** `trayectoria_camara.csv` y `vehiculos.csv` al pulsar «Detener y guardar».

Para el informe, registra video, configuración, calibración y resultados observados. Las velocidades son estimaciones académicas; valida su interpretación con la escena utilizada.